In [ ]:
# Notebook 12 — Model Development and Evaluation

## Introduction

This notebook develops and compares three machine-learning models for PEMFC voltage prediction:

- Ridge Regression
- XGBoost
- ANN/MLP

The 20 predictors confirmed during feature selection are carried forward, with Voltage as the target variable.

The models will be trained, tuned, and evaluated using a chronological framework to preserve the temporal structure of the PEMFC durability experiment and prevent data leakage.

Performance will primarily be compared using RMSE, MAE, and R².

In [ ]:
## Broad Workflow

1. Set up the modelling environment and load the engineered dataset.
2. Define the final predictors, target, and chronological data partitions.
3. Prepare model-specific preprocessing.
4. Develop and tune Ridge Regression.
5. Develop and tune XGBoost.
6. Develop and tune ANN/MLP.
7. Evaluate models using chronological validation.
8. Compare model performance using RMSE, MAE, and R².
9. Evaluate the final model(s) on the unseen holdout data.
10. Save results and summarize the final modelling decision.

In [ ]:
## F1.3 — Environment Setup

This section imports the libraries required for data preparation, model development, evaluation, visualization, and reproducibility.

In [1]:
from pathlib import Path
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("Environment setup complete.")
print("Random seed:", RANDOM_SEED)

Environment setup complete.
Random seed: 42


In [ ]:
## F1.4 — Project Paths

Project directories are defined so that the engineered dataset and modelling outputs can be accessed consistently.

In [2]:
project_root = Path.cwd().parent

data_processed_dir = project_root / "data" / "processed"
results_dir = project_root / "results" / "modeling"
models_dir = project_root / "models"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Processed data:", data_processed_dir)
print("Results directory:", results_dir)
print("Models directory:", models_dir)

Project root: C:\Users\usman\Desktop\PEMFC_Dissertation
Processed data: C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
Results directory: C:\Users\usman\Desktop\PEMFC_Dissertation\results\modeling
Models directory: C:\Users\usman\Desktop\PEMFC_Dissertation\models


In [ ]:
## F1.5 — Identify Engineered Dataset

The available processed CSV files are inspected before loading the modelling dataset to ensure that the correct engineered dataset is used.

In [3]:
processed_csv_files = sorted(data_processed_dir.glob("*.csv"))

print(f"CSV files found: {len(processed_csv_files)}\n")

for i, file_path in enumerate(processed_csv_files, start=1):
    print(f"{i}. {file_path.name}")

CSV files found: 7

1. dataset_structure_summary.csv
2. ml_variable_classification.csv
3. operational_cleaned.csv
4. operational_merged_raw.csv
5. overall_summary.csv
6. pemfc_feature_engineered.csv
7. variable_dictionary.csv


In [ ]:
## F1.6 — Load Engineered Dataset

The engineered PEMFC dataset is loaded and its basic structure is inspected before modelling.

In [4]:
engineered_data_path = data_processed_dir / "pemfc_feature_engineered.csv"

modeling_df = pd.read_csv(engineered_data_path)

print("Dataset loaded successfully.")
print("Shape:", modeling_df.shape)
print("Number of columns:", modeling_df.shape[1])

Dataset loaded successfully.
Shape: (3629680, 24)
Number of columns: 24


In [5]:
print("Columns:\n")

for column in modeling_df.columns:
    print(column)

Columns:

operating_hour
time
current
voltage
power
pressure_anode_inlet
pressure_anode_outlet
pressure_cathode_inlet
pressure_cathode_outlet
temp_anode_endplate
temp_anode_dewpoint_water
temp_anode_inlet
temp_anode_outlet
temp_cathode_dewpoint_water
temp_cathode_inlet
temp_cathode_outlet
total_anode_stack_flow
total_cathode_stack_flow
anode_pressure_diff
cathode_pressure_diff
anode_temp_diff
cathode_temp_diff
anode_dewpoint_offset
cathode_dewpoint_offset


In [ ]:
## F1.7 — Modelling Specification

Voltage is defined as the prediction target. The 20 predictors confirmed by XGBoost-Boruta in Notebook 11 are carried forward into modelling.

In [5]:
target_column = "voltage"
stage_column = "operating_hour"

selected_features = [
    "anode_pressure_diff",
    "anode_temp_diff",
    "cathode_pressure_diff",
    "current",
    "pressure_anode_outlet",
    "pressure_cathode_outlet",
    "temp_anode_dewpoint_water",
    "temp_anode_endplate",
    "temp_cathode_inlet",
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
    "cathode_dewpoint_offset",
    "temp_cathode_dewpoint_water",
    "temp_anode_outlet",
    "pressure_cathode_inlet",
    "temp_cathode_outlet",
    "pressure_anode_inlet",
    "temp_anode_inlet",
    "cathode_temp_diff",
    "anode_dewpoint_offset"
]

print("Target:", target_column)
print("Stage identifier:", stage_column)
print("Selected predictors:", len(selected_features))

Target: voltage
Stage identifier: operating_hour
Selected predictors: 20


In [ ]:
## F1.8 — Data Integrity Checks

The modelling variables are checked for availability, missing values, non-finite values, data types, and durability-stage coverage before model development.

In [6]:
required_columns = [stage_column] + selected_features + [target_column]

missing_columns = [
    column for column in required_columns
    if column not in modeling_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing from dataset: {missing_columns}"
    )

print("All required modelling columns are present.")

All required modelling columns are present.


In [7]:
modelling_check_df = modeling_df[required_columns]

print("Missing values:")
print(modelling_check_df.isna().sum().sort_values(ascending=False).head(10))

numeric_values = modelling_check_df[selected_features + [target_column]].to_numpy()

print("\nNon-finite values:", np.sum(~np.isfinite(numeric_values)))

Missing values:
operating_hour               0
anode_pressure_diff          0
anode_temp_diff              0
cathode_pressure_diff        0
current                      0
pressure_anode_outlet        0
pressure_cathode_outlet      0
temp_anode_dewpoint_water    0
temp_anode_endplate          0
temp_cathode_inlet           0
dtype: int64

Non-finite values: 0


In [8]:
print("Data types:\n")
print(modelling_check_df.dtypes)

Data types:

operating_hour                   int64
anode_pressure_diff            float64
anode_temp_diff                float64
cathode_pressure_diff          float64
current                        float64
pressure_anode_outlet          float64
pressure_cathode_outlet        float64
temp_anode_dewpoint_water      float64
temp_anode_endplate            float64
temp_cathode_inlet             float64
total_anode_stack_flow         float64
total_cathode_stack_flow       float64
cathode_dewpoint_offset        float64
temp_cathode_dewpoint_water    float64
temp_anode_outlet              float64
pressure_cathode_inlet         float64
temp_cathode_outlet            float64
pressure_anode_inlet           float64
temp_anode_inlet               float64
cathode_temp_diff              float64
anode_dewpoint_offset          float64
voltage                        float64
dtype: object


In [9]:
available_stages = sorted(
    modeling_df[stage_column]
    .dropna()
    .unique()
    .tolist()
)

print("Available operating-hour stages:")
print(available_stages)

print("\nNumber of stages:", len(available_stages))

Available operating-hour stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

Number of stages: 20


In [10]:
stage_counts = (
    modeling_df
    .groupby(stage_column)
    .size()
    .reset_index(name="Row_Count")
)

stage_counts

,operating_hour,Row_Count
0,50,179360
1,100,179360
2,150,179360
3,200,179360
4,250,179360
5,300,179360
6,350,179360
7,400,179360
8,450,179360
9,500,179360


In [ ]:
## F1.7 — Variable Role Definition

Before constructing the modelling dataset, variables that should not be used as predictors are explicitly identified.

- **Voltage** is the prediction target.
- **Operating hour** represents the durability stage and is retained for chronological splitting and stage-wise evaluation, but is not used as a predictor.
- **Time** represents measurement time within each durability-stage dataset and is not included as a predictor.
- **Power** is excluded because Power = Voltage × Current, meaning that it contains direct information from the target and would introduce target leakage.

The remaining 20 variables constitute the predictor set established during feature selection.

In [6]:
target_column = "voltage"
stage_column = "operating_hour"
time_column = "time"
leakage_column = "power"

non_predictor_columns = [
    stage_column,
    time_column,
    target_column,
    leakage_column
]

print("Target:", target_column)
print("Chronological identifier:", stage_column)
print("Excluded time variable:", time_column)
print("Excluded leakage variable:", leakage_column)

Target: voltage
Chronological identifier: operating_hour
Excluded time variable: time
Excluded leakage variable: power


In [7]:
predictor_columns = [
    column for column in modeling_df.columns
    if column not in non_predictor_columns
]

print("Total dataset columns:", modeling_df.shape[1])
print("Non-predictor columns:", len(non_predictor_columns))
print("Predictor columns:", len(predictor_columns))

print("\nPredictors:")
for predictor in predictor_columns:
    print(predictor)

Total dataset columns: 24
Non-predictor columns: 4
Predictor columns: 20

Predictors:
current
pressure_anode_inlet
pressure_anode_outlet
pressure_cathode_inlet
pressure_cathode_outlet
temp_anode_endplate
temp_anode_dewpoint_water
temp_anode_inlet
temp_anode_outlet
temp_cathode_dewpoint_water
temp_cathode_inlet
temp_cathode_outlet
total_anode_stack_flow
total_cathode_stack_flow
anode_pressure_diff
cathode_pressure_diff
anode_temp_diff
cathode_temp_diff
anode_dewpoint_offset
cathode_dewpoint_offset


In [8]:
power_reconstructed = (
    modeling_df["voltage"] *
    modeling_df["current"]
)

power_difference = (
    modeling_df["power"] -
    power_reconstructed
)

print(
    "Maximum absolute difference between recorded and V × I:",
    power_difference.abs().max()
)

print(
    "Mean absolute difference:",
    power_difference.abs().mean()
)

Maximum absolute difference between recorded and V × I: 0.006708119999998985
Mean absolute difference: 0.0024365676921932443


In [9]:
time_stage_summary = (
    modeling_df
    .groupby("operating_hour")["time"]
    .agg(
        Min_Time="min",
        Max_Time="max",
        Number_of_Values="count"
    )
    .reset_index()
)

time_stage_summary

,operating_hour,Min_Time,Max_Time,Number_of_Values
0,50,1.761,179360.761,179360
1,100,1.226,179360.226,179360
2,150,1.765,179360.765,179360
3,200,1.280,179360.280,179360
4,250,1.740,179360.740,179360
5,300,1.738,179360.738,179360
6,350,1.159,179360.159,179360
7,400,1.650,179360.650,179360
8,450,1.778,179360.778,179360
9,500,1.278,179360.278,179360


In [ ]:
## F1.8 — Modelling Data Sample Inspection

A sample of the engineered dataset is inspected to understand its row-level structure, measurement sequence, and how observations are organised within and across durability stages before constructing the modelling framework.

In [16]:
modeling_df.head(10)

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.761,0.0,0.9375,0.0,109.901303,110.327250,109.800128,108.792114,83.128159,...,69.673592,56.337543,0.07,0.291,-0.425947,1.008014,-32.498287,-13.336049,17.486717,5.209305
1,50,2.761,0.0,0.9375,0.0,110.103653,110.327250,109.800128,108.893387,83.078773,...,69.661247,56.324917,0.07,0.291,-0.223597,0.906741,-32.515045,-13.336330,17.437339,5.222213
2,50,3.761,0.0,0.9372,0.0,110.306003,110.327250,109.800128,108.792114,83.078773,...,69.685936,56.299664,0.07,0.291,-0.021247,1.008014,-32.476250,-13.386272,17.360397,5.199318
3,50,4.761,0.0,0.9375,0.0,110.103653,110.327250,109.800128,108.792114,83.103462,...,69.685936,56.324917,0.07,0.291,-0.223597,1.008014,-32.552078,-13.361019,17.461452,5.246902
4,50,5.761,0.0,0.9372,0.0,109.901303,110.327250,109.698953,108.893387,83.091118,...,69.661247,56.249161,0.07,0.291,-0.425947,0.805566,-32.515930,-13.412086,17.412075,5.171715
5,50,6.761,0.0,0.9375,0.0,109.901303,110.327250,109.800128,108.792114,83.066429,...,69.661247,56.249161,0.07,0.291,-0.425947,1.008014,-32.515045,-13.412086,17.424419,5.196960
6,50,7.761,0.0,0.9375,0.0,110.204828,110.327250,109.800128,108.893387,83.189880,...,69.685936,56.236534,0.07,0.291,-0.122422,0.906741,-32.490349,-13.449402,17.348045,5.234276
7,50,8.761,0.0,0.9372,0.0,109.901303,110.327250,109.698953,108.893387,83.066429,...,69.698280,56.186031,0.07,0.291,-0.425947,0.805566,-32.526508,-13.512249,17.474949,5.208748
8,50,9.761,0.0,0.9372,0.0,110.204828,110.327250,109.800128,109.197205,83.066429,...,69.636559,56.148155,0.07,0.291,-0.122422,0.602923,-32.490349,-13.488404,17.412643,5.222779
9,50,10.761,0.0,0.9372,0.0,109.901303,110.125047,109.800128,109.197205,83.041740,...,69.636559,56.135529,0.07,0.291,-0.223744,0.602923,-32.416283,-13.501030,17.364418,5.197525


In [17]:
stage_sample = (
    modeling_df
    .groupby("operating_hour", group_keys=False)
    .head(3)
)

stage_sample

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.761,0.0000,0.9375,0.0,109.901303,110.327250,109.800128,108.792114,83.128159,...,69.673592,56.337543,0.070,0.291,-0.425947,1.008014,-32.498287,-13.336049,17.486717,5.209305
1,50,2.761,0.0000,0.9375,0.0,110.103653,110.327250,109.800128,108.893387,83.078773,...,69.661247,56.324917,0.070,0.291,-0.223597,0.906741,-32.515045,-13.336330,17.437339,5.222213
2,50,3.761,0.0000,0.9372,0.0,110.306003,110.327250,109.800128,108.792114,83.078773,...,69.685936,56.299664,0.070,0.291,-0.021247,1.008014,-32.476250,-13.386272,17.360397,5.199318
179360,100,1.226,0.0000,0.9433,0.0,109.901303,110.529453,109.395428,108.792114,83.004883,...,70.225243,56.262810,0.070,0.291,-0.628150,0.603314,-31.771306,-13.962433,14.069722,5.670174
179361,100,2.226,0.0000,0.9430,0.0,110.204828,110.529453,109.597778,108.792114,82.980629,...,70.212616,56.250183,0.070,0.291,-0.324625,0.805664,-31.747501,-13.962433,14.058525,5.619667
179362,100,3.226,0.0000,0.9427,0.0,109.901303,110.529453,109.698953,108.792114,83.004883,...,70.288368,56.275436,0.070,0.291,-0.628150,0.906839,-31.782776,-14.012932,14.120255,5.733299
358720,150,1.765,0.0000,0.9465,0.0,109.901303,110.428352,109.597778,108.792114,82.925072,...,70.037560,57.361057,0.070,0.291,-0.527049,0.805664,-26.753251,-12.676503,13.811951,5.545708
358721,150,2.765,0.0000,0.9462,0.0,110.204828,110.125047,109.597778,108.589569,82.925072,...,70.062813,57.348137,0.070,0.291,0.079781,1.008209,-26.765278,-12.714676,13.799328,5.583588
358722,150,3.765,0.0000,0.9459,0.0,110.103653,110.529453,109.597778,108.589569,82.937149,...,70.075439,57.322296,0.070,0.291,-0.425800,1.008209,-26.791729,-12.753143,13.849831,5.545707
538080,200,1.280,0.0000,0.9398,0.0,110.204828,110.428352,110.002478,108.792114,82.868393,...,69.894188,58.030148,0.084,0.349,-0.223524,1.210364,-31.107308,-11.864040,15.276158,5.326477


In [18]:
sample_stage = 50

sequential_sample = (
    modeling_df.loc[
        modeling_df["operating_hour"] == sample_stage
    ]
    .head(20)
)

sequential_sample

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,...,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.761,0.0000,0.9375,0.00,109.901303,110.327250,109.800128,108.792114,83.128159,...,69.673592,56.337543,0.07,0.291,-0.425947,1.008014,-32.498287,-13.336049,17.486717,5.209305
1,50,2.761,0.0000,0.9375,0.00,110.103653,110.327250,109.800128,108.893387,83.078773,...,69.661247,56.324917,0.07,0.291,-0.223597,0.906741,-32.515045,-13.336330,17.437339,5.222213
2,50,3.761,0.0000,0.9372,0.00,110.306003,110.327250,109.800128,108.792114,83.078773,...,69.685936,56.299664,0.07,0.291,-0.021247,1.008014,-32.476250,-13.386272,17.360397,5.199318
3,50,4.761,0.0000,0.9375,0.00,110.103653,110.327250,109.800128,108.792114,83.103462,...,69.685936,56.324917,0.07,0.291,-0.223597,1.008014,-32.552078,-13.361019,17.461452,5.246902
4,50,5.761,0.0000,0.9372,0.00,109.901303,110.327250,109.698953,108.893387,83.091118,...,69.661247,56.249161,0.07,0.291,-0.425947,0.805566,-32.515930,-13.412086,17.412075,5.171715
5,50,6.761,0.0000,0.9375,0.00,109.901303,110.327250,109.800128,108.792114,83.066429,...,69.661247,56.249161,0.07,0.291,-0.425947,1.008014,-32.515045,-13.412086,17.424419,5.196960
6,50,7.761,0.0000,0.9375,0.00,110.204828,110.327250,109.800128,108.893387,83.189880,...,69.685936,56.236534,0.07,0.291,-0.122422,0.906741,-32.490349,-13.449402,17.348045,5.234276
7,50,8.761,0.0000,0.9372,0.00,109.901303,110.327250,109.698953,108.893387,83.066429,...,69.698280,56.186031,0.07,0.291,-0.425947,0.805566,-32.526508,-13.512249,17.474949,5.208748
8,50,9.761,0.0000,0.9372,0.00,110.204828,110.327250,109.800128,109.197205,83.066429,...,69.636559,56.148155,0.07,0.291,-0.122422,0.602923,-32.490349,-13.488404,17.412643,5.222779
9,50,10.761,0.0000,0.9372,0.00,109.901303,110.125047,109.800128,109.197205,83.041740,...,69.636559,56.135529,0.07,0.291,-0.223744,0.602923,-32.416283,-13.501030,17.364418,5.197525


In [10]:
ordering_check = (
    modeling_df[
        ["operating_hour", "time"]
    ]
    .copy()
)

stage_order_valid = (
    ordering_check["operating_hour"]
    .diff()
    .fillna(0)
    >= 0
).all()

time_order_by_stage = (
    modeling_df
    .groupby("operating_hour")["time"]
    .apply(lambda x: x.is_monotonic_increasing)
)

print("Operating-hour order is chronological:", stage_order_valid)

print("\nTime monotonically increases within each stage:")
print(time_order_by_stage)

Operating-hour order is chronological: True

Time monotonically increases within each stage:
operating_hour
50      True
100     True
150     True
200     True
250     True
300     True
350     True
400     True
450     True
500     True
550     True
600     True
650     True
700     True
750     True
800     True
850     True
900     True
950     True
1000    True
Name: time, dtype: bool


In [ ]:
## F1.9 — Chronological Modelling Design

The PEMFC durability data are divided chronologically to prevent information from later operating stages influencing model development.

Stages from 50 h to 850 h are used for model development and chronological validation, while 900 h, 950 h, and 1000 h are reserved as the final unseen holdout set.

The holdout stages will not be used during model fitting, hyperparameter tuning, or model selection.

In [11]:
development_stages = [
    50, 100, 150, 200, 250,
    300, 350, 400, 450, 500,
    550, 600, 650, 700, 750,
    800, 850
]

holdout_stages = [
    900, 950, 1000
]

print("Development stages:")
print(development_stages)

print("\nFinal holdout stages:")
print(holdout_stages)

Development stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850]

Final holdout stages:
[900, 950, 1000]


In [12]:
overlap = set(development_stages).intersection(holdout_stages)

print("Overlap between development and holdout:", overlap)

assert len(overlap) == 0, "Development and holdout stages overlap."

Overlap between development and holdout: set()


In [13]:
defined_stages = sorted(
    development_stages + holdout_stages
)

available_stages = sorted(
    modeling_df["operating_hour"]
    .dropna()
    .unique()
    .tolist()
)

print("Available stages:")
print(available_stages)

print("\nDefined stages:")
print(defined_stages)

assert available_stages == defined_stages, (
    "Defined stages do not match the stages available in the dataset."
)

Available stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

Defined stages:
[50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]


In [ ]:
## F1.10 — Development and Holdout Dataset Construction

The engineered dataset is separated into a model-development dataset and an untouched final holdout dataset using the predefined durability stages.

In [14]:
development_df = modeling_df[
    modeling_df["operating_hour"].isin(development_stages)
].copy()

holdout_df = modeling_df[
    modeling_df["operating_hour"].isin(holdout_stages)
].copy()

print("Development dataset shape:", development_df.shape)
print("Holdout dataset shape:", holdout_df.shape)

print("\nDevelopment stages:")
print(sorted(development_df["operating_hour"].unique()))

print("\nHoldout stages:")
print(sorted(holdout_df["operating_hour"].unique()))

Development dataset shape: (3049120, 24)
Holdout dataset shape: (580560, 24)

Development stages:
[np.int64(50), np.int64(100), np.int64(150), np.int64(200), np.int64(250), np.int64(300), np.int64(350), np.int64(400), np.int64(450), np.int64(500), np.int64(550), np.int64(600), np.int64(650), np.int64(700), np.int64(750), np.int64(800), np.int64(850)]

Holdout stages:
[np.int64(900), np.int64(950), np.int64(1000)]


In [ ]:
## F1.11 — Final Modelling Variables

The final predictor matrix contains the 20 Boruta-confirmed predictors.

Voltage is used as the target variable.

Operating hour, time, and power are retained in the source dataset but excluded from the predictor matrix.

In [28]:
predictor_columns = [
    "current",
    "pressure_anode_inlet",
    "pressure_anode_outlet",
    "pressure_cathode_inlet",
    "pressure_cathode_outlet",
    "temp_anode_endplate",
    "temp_anode_dewpoint_water",
    "temp_anode_inlet",
    "temp_anode_outlet",
    "temp_cathode_dewpoint_water",
    "temp_cathode_inlet",
    "temp_cathode_outlet",
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset"
]

target_column = "voltage"

print("Number of predictors:", len(predictor_columns))
print("Target:", target_column)

Number of predictors: 20
Target: voltage


In [29]:
X_development = development_df[predictor_columns].copy()
y_development = development_df[target_column].copy()

X_holdout = holdout_df[predictor_columns].copy()
y_holdout = holdout_df[target_column].copy()

print("X development shape:", X_development.shape)
print("y development shape:", y_development.shape)

print("\nX holdout shape:", X_holdout.shape)
print("y holdout shape:", y_holdout.shape)

X development shape: (3049120, 20)
y development shape: (3049120,)

X holdout shape: (580560, 20)
y holdout shape: (580560,)


In [ ]:
## F1.12 — Chronological Validation Folds

Expanding chronological folds are used within the development dataset.

Each fold trains on earlier durability stages and validates on later unseen stages. This preserves the temporal progression of PEMFC aging during model development.

In [30]:
chronological_folds = {
    "Fold_1": {
        "train_stages": [
            50, 100, 150, 200, 250,
            300, 350, 400, 450
        ],
        "validation_stages": [500, 550]
    },

    "Fold_2": {
        "train_stages": [
            50, 100, 150, 200, 250,
            300, 350, 400, 450, 500, 550
        ],
        "validation_stages": [600, 650]
    },

    "Fold_3": {
        "train_stages": [
            50, 100, 150, 200, 250,
            300, 350, 400, 450, 500,
            550, 600, 650
        ],
        "validation_stages": [700, 750]
    },

    "Fold_4": {
        "train_stages": [
            50, 100, 150, 200, 250,
            300, 350, 400, 450, 500,
            550, 600, 650, 700, 750
        ],
        "validation_stages": [800, 850]
    }
}

for fold_name, fold_info in chronological_folds.items():
    print(f"\n{fold_name}")
    print("Training stages:", fold_info["train_stages"])
    print("Validation stages:", fold_info["validation_stages"])


Fold_1
Training stages: [50, 100, 150, 200, 250, 300, 350, 400, 450]
Validation stages: [500, 550]

Fold_2
Training stages: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550]
Validation stages: [600, 650]

Fold_3
Training stages: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650]
Validation stages: [700, 750]

Fold_4
Training stages: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750]
Validation stages: [800, 850]


In [33]:
# ============================================================
# Restore Chronological Fold Definitions
# ============================================================

boruta_outer_folds = {
    "Fold_1": {
        "train_labels": list(range(50, 451, 50)),
        "validation_labels": [500, 550]
    },
    "Fold_2": {
        "train_labels": list(range(50, 551, 50)),
        "validation_labels": [600, 650]
    },
    "Fold_3": {
        "train_labels": list(range(50, 651, 50)),
        "validation_labels": [700, 750]
    },
    "Fold_4": {
        "train_labels": list(range(50, 751, 50)),
        "validation_labels": [800, 850]
    }
}

print("Chronological folds restored:", len(boruta_outer_folds))

for fold_name, fold_info in boruta_outer_folds.items():
    print(
        fold_name,
        "| Train:",
        f"{min(fold_info['train_labels'])}-{max(fold_info['train_labels'])}",
        "| Validation:",
        fold_info["validation_labels"]
    )

Chronological folds restored: 4
Fold_1 | Train: 50-450 | Validation: [500, 550]
Fold_2 | Train: 50-550 | Validation: [600, 650]
Fold_3 | Train: 50-650 | Validation: [700, 750]
Fold_4 | Train: 50-750 | Validation: [800, 850]


In [31]:
for fold_name, fold_info in chronological_folds.items():

    train_stages = fold_info["train_stages"]
    validation_stages = fold_info["validation_stages"]

    train_max = max(train_stages)
    validation_min = min(validation_stages)

    overlap = set(train_stages).intersection(validation_stages)

    print(
        fold_name,
        "| Train end:", train_max,
        "| Validation start:", validation_min,
        "| Overlap:", len(overlap)
    )

    assert len(overlap) == 0
    assert train_max < validation_min

Fold_1 | Train end: 450 | Validation start: 500 | Overlap: 0
Fold_2 | Train end: 550 | Validation start: 600 | Overlap: 0
Fold_3 | Train end: 650 | Validation start: 700 | Overlap: 0
Fold_4 | Train end: 750 | Validation start: 800 | Overlap: 0


In [ ]:
## F1.13 — Ridge Regression

Ridge Regression is implemented as the regularized linear baseline.

Because Ridge is sensitive to predictor scale, the predictors are standardized using parameters learned from the training data only. Validation data are transformed using the corresponding training-fold scaler to prevent information leakage.

In [19]:
import gc
import time

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


print("Ridge modelling utilities ready.")

Ridge modelling utilities ready.


In [ ]:
### F1.13.1 — Baseline Ridge Configuration

Ridge Regression is used as the regularized linear baseline.

Because the predictors have different units and scales, StandardScaler is
applied before Ridge. Scaling parameters are learned from training data only
to prevent chronological leakage.

A baseline α = 1.0 is used at this stage; hyperparameter tuning is performed
separately later.

In [21]:
# ============================================================
# F1.13.1 — Baseline Ridge Configuration
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# Baseline Ridge settings
# ------------------------------------------------------------

ridge_baseline_config = {
    "alpha": 1.0,
    "fit_intercept": True,
    "solver": "auto"
}

# ------------------------------------------------------------
# Build leakage-safe modelling pipeline
# ------------------------------------------------------------

ridge_baseline_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(
            **ridge_baseline_config
        )
    )
])

# ------------------------------------------------------------
# Report configuration
# ------------------------------------------------------------

print("Baseline Ridge Configuration")
print("============================")

for parameter, value in ridge_baseline_config.items():
    print(f"{parameter}: {value}")

print("\nPipeline")
print("--------")
print("1. StandardScaler")
print("2. Ridge Regression")

print("\nStandardization fitted within training data only: True")
print("Hyperparameter tuning performed at this stage: False")
print("Role: Regularized linear baseline")

Baseline Ridge Configuration
alpha: 1.0
fit_intercept: True
solver: auto

Pipeline
--------
1. StandardScaler
2. Ridge Regression

Standardization fitted within training data only: True
Hyperparameter tuning performed at this stage: False
Role: Regularized linear baseline


In [ ]:
### F1.13.2 — Chronological Ridge Evaluation Function

A reusable function is defined to train and evaluate Ridge consistently across
the chronological folds.

For each fold, scaling and model fitting use training data only. Performance is
evaluated on the corresponding unseen later-stage validation data using RMSE,
MAE and R².

In [23]:
# ============================================================
# F1.13.2 — Chronological Ridge Evaluation Function
# ============================================================

from sklearn.base import clone
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

def evaluate_ridge_fold(
    X_train,
    y_train,
    X_validation,
    y_validation,
    feature_set,
    fold_name
):
    """
    Train and evaluate the baseline Ridge model on one
    chronological fold.
    """

    # Select predictors
    X_train_selected = X_train[feature_set]
    X_validation_selected = X_validation[feature_set]

    # Fresh pipeline for this fold
    model = clone(ridge_baseline_pipeline)

    # Train only on historical fold data
    model.fit(
        X_train_selected,
        y_train
    )

    # Predict unseen later-stage data
    y_pred = model.predict(
        X_validation_selected
    )

    # Performance metrics
    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_validation,
        y_pred
    )

    r2 = r2_score(
        y_validation,
        y_pred
    )

    results = {
        "fold": fold_name,
        "n_features": len(feature_set),
        "train_observations": len(X_train),
        "validation_observations": len(X_validation),
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

    return results, model

In [ ]:
### F1.13.3 — Baseline Ridge Across Chronological Folds

Baseline Ridge is evaluated across the four expanding chronological folds.

Each fold is trained only on earlier durability stages and evaluated on the
corresponding unseen later-stage validation stages.

In [34]:
# ============================================================
# F1.13.3 — Baseline Ridge Across Chronological Folds
# ============================================================

ridge_fold_results = []
ridge_fold_models = {}

ridge_feature_set = boruta_candidate_predictors.copy()

print("Baseline Ridge Chronological Evaluation")
print("=======================================")

for fold_name, fold_info in boruta_outer_folds.items():

    train_labels = fold_info["train_labels"]
    validation_labels = fold_info["validation_labels"]

    # --------------------------------------------------------
    # Build fold-specific training and validation datasets
    # --------------------------------------------------------

    train_df = development_df[
        development_df["operating_hour"].isin(
            train_labels
        )
    ]

    validation_df = development_df[
        development_df["operating_hour"].isin(
            validation_labels
        )
    ]

    X_train = train_df[ridge_feature_set]
    y_train = train_df["voltage"]

    X_validation = validation_df[ridge_feature_set]
    y_validation = validation_df["voltage"]

    # --------------------------------------------------------
    # Evaluate Ridge
    # --------------------------------------------------------

    fold_result, fitted_model = evaluate_ridge_fold(
        X_train=X_train,
        y_train=y_train,
        X_validation=X_validation,
        y_validation=y_validation,
        feature_set=ridge_feature_set,
        fold_name=fold_name
    )

    ridge_fold_results.append(
        fold_result
    )

    ridge_fold_models[
        fold_name
    ] = fitted_model

    print(f"\n{fold_name}")
    print("-" * len(fold_name))

    print(
        "Train labels:",
        f"{min(train_labels)}–{max(train_labels)}"
    )

    print(
        "Validation labels:",
        validation_labels
    )

    print(
        f"RMSE: {fold_result['rmse']:.6f} V"
    )

    print(
        f"MAE:  {fold_result['mae']:.6f} V"
    )

    print(
        f"R²:   {fold_result['r2']:.6f}"
    )


# ------------------------------------------------------------
# Combine results
# ------------------------------------------------------------

ridge_chronological_results = pd.DataFrame(
    ridge_fold_results
)

print("\nChronological Ridge Summary")
print("===========================")

print(
    ridge_chronological_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Baseline Ridge Chronological Evaluation

Fold_1
------
Train labels: 50–450
Validation labels: [500, 550]
RMSE: 0.018224 V
MAE:  0.014882 V
R²:   0.965270

Fold_2
------
Train labels: 50–550
Validation labels: [600, 650]
RMSE: 0.016332 V
MAE:  0.012935 V
R²:   0.973288

Fold_3
------
Train labels: 50–650
Validation labels: [700, 750]
RMSE: 0.019800 V
MAE:  0.016798 V
R²:   0.962488

Fold_4
------
Train labels: 50–750
Validation labels: [800, 850]
RMSE: 0.014051 V
MAE:  0.010049 V
R²:   0.979747

Chronological Ridge Summary
  fold  n_features  train_observations  validation_observations     rmse      mae       r2
Fold_1          20             1614240                   358720 0.018224 0.014882 0.965270
Fold_2          20             1972960                   358720 0.016332 0.012935 0.973288
Fold_3          20             2331680                   358720 0.019800 0.016798 0.962488
Fold_4          20             2690400                   358720 0.014051 0.010049 0.979747


In [ ]:
### F1.13.4 — Ridge Performance Summary

Ridge performance is summarized across the chronological folds to establish
the overall linear baseline and assess variation between validation periods.

In [35]:
# ============================================================
# F1.13.4 — Ridge Performance Summary
# ============================================================

ridge_performance_summary = pd.DataFrame({
    "metric": ["RMSE", "MAE", "R²"],
    "mean": [
        ridge_chronological_results["rmse"].mean(),
        ridge_chronological_results["mae"].mean(),
        ridge_chronological_results["r2"].mean()
    ],
    "std": [
        ridge_chronological_results["rmse"].std(),
        ridge_chronological_results["mae"].std(),
        ridge_chronological_results["r2"].std()
    ],
    "minimum": [
        ridge_chronological_results["rmse"].min(),
        ridge_chronological_results["mae"].min(),
        ridge_chronological_results["r2"].min()
    ],
    "maximum": [
        ridge_chronological_results["rmse"].max(),
        ridge_chronological_results["mae"].max(),
        ridge_chronological_results["r2"].max()
    ]
})

print("Ridge Chronological Performance Summary")
print("=======================================")

print(
    ridge_performance_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Ridge Chronological Performance Summary
metric     mean      std  minimum  maximum
  RMSE 0.017102 0.002479 0.014051 0.019800
   MAE 0.013666 0.002881 0.010049 0.016798
    R² 0.970199 0.007841 0.962488 0.979747


In [ ]:
### F1.13.4 — Ridge Hyperparameter Tuning Framework

Ridge regularization strength (α) is tuned within each outer training fold
using an inner chronological validation split.

This prevents the outer validation stages from influencing hyperparameter
selection and preserves chronological evaluation integrity.

In [36]:
# ============================================================
# F1.13.4 — Ridge Hyperparameter Tuning Framework
# ============================================================

ridge_alpha_grid = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0
]

print("Ridge Alpha Search Space")
print("========================")

for alpha in ridge_alpha_grid:
    print(f"α = {alpha}")

print("\nNumber of candidate values:", len(ridge_alpha_grid))
print("Selection metric: RMSE")
print("Tuning strategy: Inner chronological validation")
print("Outer validation used for tuning: False")
print("Final test used for tuning: False")

Ridge Alpha Search Space
α = 0.001
α = 0.01
α = 0.1
α = 1.0
α = 10.0
α = 100.0
α = 1000.0

Number of candidate values: 7
Selection metric: RMSE
Tuning strategy: Inner chronological validation
Outer validation used for tuning: False
Final test used for tuning: False


In [ ]:
### F1.13.5 — Inner Chronological Splits

Within each outer training fold, the latest two available stages are used for
inner validation and all earlier stages are used for inner training.

This allows α to be selected without using the outer validation stages.

In [37]:
# ============================================================
# F1.13.5 — Define Inner Chronological Splits
# ============================================================

ridge_inner_splits = {}

print("Ridge Inner Chronological Tuning Splits")
print("=======================================")

for fold_name, fold_info in boruta_outer_folds.items():

    outer_train_labels = fold_info["train_labels"]

    inner_validation_labels = outer_train_labels[-2:]
    inner_train_labels = outer_train_labels[:-2]

    ridge_inner_splits[fold_name] = {
        "inner_train_labels": inner_train_labels,
        "inner_validation_labels": inner_validation_labels
    }

    print(f"\n{fold_name}")
    print("-" * len(fold_name))
    print("Inner train labels:", inner_train_labels)
    print("Inner validation labels:", inner_validation_labels)

    print(
        "Chronological ordering preserved:",
        max(inner_train_labels) < min(inner_validation_labels)
    )

    print(
        "Outer validation excluded:",
        not set(fold_info["validation_labels"]).intersection(
            inner_train_labels + inner_validation_labels
        )
    )

Ridge Inner Chronological Tuning Splits

Fold_1
------
Inner train labels: [50, 100, 150, 200, 250, 300, 350]
Inner validation labels: [400, 450]
Chronological ordering preserved: True
Outer validation excluded: True

Fold_2
------
Inner train labels: [50, 100, 150, 200, 250, 300, 350, 400, 450]
Inner validation labels: [500, 550]
Chronological ordering preserved: True
Outer validation excluded: True

Fold_3
------
Inner train labels: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550]
Inner validation labels: [600, 650]
Chronological ordering preserved: True
Outer validation excluded: True

Fold_4
------
Inner train labels: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650]
Inner validation labels: [700, 750]
Chronological ordering preserved: True
Outer validation excluded: True


In [ ]:
### F1.13.6 — Ridge Alpha Tuning

Each candidate α value is evaluated on the inner chronological split of every
outer fold.

The α with the lowest inner-validation RMSE is selected separately for each
outer fold.

In [38]:
# ============================================================
# F1.13.6 — Ridge Alpha Tuning
# ============================================================

ridge_tuning_records = []
ridge_selected_alphas = {}

print("Ridge Alpha Tuning")
print("==================")

for fold_name, split_info in ridge_inner_splits.items():

    inner_train_labels = split_info["inner_train_labels"]
    inner_validation_labels = split_info["inner_validation_labels"]

    inner_train_df = development_df[
        development_df["operating_hour"].isin(inner_train_labels)
    ]

    inner_validation_df = development_df[
        development_df["operating_hour"].isin(inner_validation_labels)
    ]

    X_inner_train = inner_train_df[boruta_candidate_predictors]
    y_inner_train = inner_train_df["voltage"]

    X_inner_validation = inner_validation_df[boruta_candidate_predictors]
    y_inner_validation = inner_validation_df["voltage"]

    fold_results = []

    print(f"\n{fold_name}")
    print("-" * len(fold_name))

    for alpha in ridge_alpha_grid:

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(
                alpha=alpha,
                fit_intercept=True,
                solver="auto"
            ))
        ])

        model.fit(
            X_inner_train,
            y_inner_train
        )

        y_pred = model.predict(
            X_inner_validation
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_inner_validation,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_inner_validation,
            y_pred
        )

        r2 = r2_score(
            y_inner_validation,
            y_pred
        )

        result = {
            "fold": fold_name,
            "alpha": alpha,
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }

        fold_results.append(result)
        ridge_tuning_records.append(result)

        print(
            f"α={alpha:<7} | "
            f"RMSE={rmse:.6f} V | "
            f"MAE={mae:.6f} V | "
            f"R²={r2:.6f}"
        )

    fold_results_df = pd.DataFrame(fold_results)

    best_row = fold_results_df.loc[
        fold_results_df["rmse"].idxmin()
    ]

    best_alpha = best_row["alpha"]

    ridge_selected_alphas[
        fold_name
    ] = best_alpha

    print(
        f"Selected α for {fold_name}: "
        f"{best_alpha}"
    )


ridge_tuning_results = pd.DataFrame(
    ridge_tuning_records
)

print("\nSelected Alpha Values")
print("=====================")

for fold_name, alpha in ridge_selected_alphas.items():
    print(f"{fold_name}: α = {alpha}")

Ridge Alpha Tuning

Fold_1
------
α=0.001   | RMSE=0.047113 V | MAE=0.037729 V | R²=0.766830
α=0.01    | RMSE=0.047114 V | MAE=0.037729 V | R²=0.766830
α=0.1     | RMSE=0.047114 V | MAE=0.037729 V | R²=0.766828
α=1.0     | RMSE=0.047115 V | MAE=0.037728 V | R²=0.766812
α=10.0    | RMSE=0.047121 V | MAE=0.037725 V | R²=0.766756
α=100.0   | RMSE=0.047058 V | MAE=0.037669 V | R²=0.767377
α=1000.0  | RMSE=0.046333 V | MAE=0.037106 V | R²=0.774493
Selected α for Fold_1: 1000.0

Fold_2
------
α=0.001   | RMSE=0.018223 V | MAE=0.014879 V | R²=0.965274
α=0.01    | RMSE=0.018223 V | MAE=0.014879 V | R²=0.965274
α=0.1     | RMSE=0.018223 V | MAE=0.014879 V | R²=0.965274
α=1.0     | RMSE=0.018224 V | MAE=0.014882 V | R²=0.965270
α=10.0    | RMSE=0.018232 V | MAE=0.014895 V | R²=0.965241
α=100.0   | RMSE=0.018252 V | MAE=0.014907 V | R²=0.965162
α=1000.0  | RMSE=0.018311 V | MAE=0.014846 V | R²=0.964940
Selected α for Fold_2: 0.001

Fold_3
------
α=0.001   | RMSE=0.016333 V | MAE=0.012934 V | R²=0

In [ ]:
### F1.13.7 — Tuned Ridge Outer Validation

The α selected within each inner chronological split is used to refit Ridge on
the complete outer training history.

The refitted model is then evaluated once on the corresponding unseen outer
validation stages.

In [39]:
# ============================================================
# F1.13.7 — Tuned Ridge Outer Validation
# ============================================================

tuned_ridge_records = []

print("Tuned Ridge Chronological Evaluation")
print("====================================")

for fold_name, fold_info in boruta_outer_folds.items():

    selected_alpha = ridge_selected_alphas[fold_name]

    train_labels = fold_info["train_labels"]
    validation_labels = fold_info["validation_labels"]

    train_df = development_df[
        development_df["operating_hour"].isin(train_labels)
    ]

    validation_df = development_df[
        development_df["operating_hour"].isin(validation_labels)
    ]

    X_train = train_df[boruta_candidate_predictors]
    y_train = train_df["voltage"]

    X_validation = validation_df[boruta_candidate_predictors]
    y_validation = validation_df["voltage"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=selected_alpha,
            fit_intercept=True,
            solver="auto"
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_validation)

    rmse = np.sqrt(
        mean_squared_error(y_validation, y_pred)
    )

    mae = mean_absolute_error(
        y_validation,
        y_pred
    )

    r2 = r2_score(
        y_validation,
        y_pred
    )

    tuned_ridge_records.append({
        "fold": fold_name,
        "selected_alpha": selected_alpha,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    })

    print(f"\n{fold_name}")
    print("-" * len(fold_name))
    print(f"Selected α: {selected_alpha}")
    print(f"RMSE: {rmse:.6f} V")
    print(f"MAE:  {mae:.6f} V")
    print(f"R²:   {r2:.6f}")


tuned_ridge_results = pd.DataFrame(
    tuned_ridge_records
)

print("\nTuned Ridge Summary")
print("===================")

print(
    tuned_ridge_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Tuned Ridge Chronological Evaluation

Fold_1
------
Selected α: 1000.0
RMSE: 0.018311 V
MAE:  0.014846 V
R²:   0.964940

Fold_2
------
Selected α: 0.001
RMSE: 0.016333 V
MAE:  0.012934 V
R²:   0.973286

Fold_3
------
Selected α: 10.0
RMSE: 0.019803 V
MAE:  0.016806 V
R²:   0.962477

Fold_4
------
Selected α: 1000.0
RMSE: 0.013785 V
MAE:  0.009955 V
R²:   0.980507

Tuned Ridge Summary
  fold  selected_alpha     rmse      mae       r2
Fold_1     1000.000000 0.018311 0.014846 0.964940
Fold_2        0.001000 0.016333 0.012934 0.973286
Fold_3       10.000000 0.019803 0.016806 0.962477
Fold_4     1000.000000 0.013785 0.009955 0.980507


In [ ]:
### F1.13.8 — Baseline vs Tuned Ridge Comparison

Baseline and tuned Ridge performance are compared across the outer
chronological folds to determine whether α tuning provides a meaningful
generalization improvement.

In [40]:
# ============================================================
# F1.13.8 — Baseline vs Tuned Ridge Comparison
# ============================================================

ridge_comparison = ridge_chronological_results[
    ["fold", "rmse", "mae", "r2"]
].copy()

ridge_comparison = ridge_comparison.rename(columns={
    "rmse": "baseline_rmse",
    "mae": "baseline_mae",
    "r2": "baseline_r2"
})

ridge_comparison = ridge_comparison.merge(
    tuned_ridge_results[
        ["fold", "selected_alpha", "rmse", "mae", "r2"]
    ].rename(columns={
        "rmse": "tuned_rmse",
        "mae": "tuned_mae",
        "r2": "tuned_r2"
    }),
    on="fold"
)

ridge_comparison["rmse_change"] = (
    ridge_comparison["tuned_rmse"]
    - ridge_comparison["baseline_rmse"]
)

ridge_comparison["mae_change"] = (
    ridge_comparison["tuned_mae"]
    - ridge_comparison["baseline_mae"]
)

ridge_comparison["r2_change"] = (
    ridge_comparison["tuned_r2"]
    - ridge_comparison["baseline_r2"]
)

print("Baseline vs Tuned Ridge")
print("=======================")

print(
    ridge_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nMean Performance")
print("================")

print(
    f"Baseline RMSE: "
    f"{ridge_comparison['baseline_rmse'].mean():.6f} V"
)

print(
    f"Tuned RMSE:    "
    f"{ridge_comparison['tuned_rmse'].mean():.6f} V"
)

print(
    f"Baseline MAE:  "
    f"{ridge_comparison['baseline_mae'].mean():.6f} V"
)

print(
    f"Tuned MAE:     "
    f"{ridge_comparison['tuned_mae'].mean():.6f} V"
)

print(
    f"Baseline R²:   "
    f"{ridge_comparison['baseline_r2'].mean():.6f}"
)

print(
    f"Tuned R²:      "
    f"{ridge_comparison['tuned_r2'].mean():.6f}"
)

Baseline vs Tuned Ridge
  fold  baseline_rmse  baseline_mae  baseline_r2  selected_alpha  tuned_rmse  tuned_mae  tuned_r2  rmse_change  mae_change  r2_change
Fold_1       0.018224      0.014882     0.965270     1000.000000    0.018311   0.014846  0.964940     0.000086   -0.000036  -0.000330
Fold_2       0.016332      0.012935     0.973288        0.001000    0.016333   0.012934  0.973286     0.000001   -0.000001  -0.000002
Fold_3       0.019800      0.016798     0.962488       10.000000    0.019803   0.016806  0.962477     0.000003    0.000008  -0.000011
Fold_4       0.014051      0.010049     0.979747     1000.000000    0.013785   0.009955  0.980507    -0.000266   -0.000094   0.000760

Mean Performance
Baseline RMSE: 0.017102 V
Tuned RMSE:    0.017058 V
Baseline MAE:  0.013666 V
Tuned MAE:     0.013635 V
Baseline R²:   0.970199
Tuned R²:      0.970303


In [ ]:
### F1.13.9 — Final Ridge Configuration Decision

Nested chronological tuning was used to assess whether changing the Ridge
regularization parameter (α) materially improved later-stage prediction.

The baseline model used α = 1.0, while inner chronological validation tested
α values from 0.001 to 1000.

Although different α values were selected across the four folds, the resulting
changes in predictive performance were negligible. Mean RMSE decreased from
0.017102 V to 0.017058 V, corresponding to an improvement of approximately
0.26%, while mean R² changed from 0.970199 to 0.970303.

The selected α values also varied substantially between folds despite very
similar validation errors, indicating that Ridge performance was relatively
insensitive to the regularization strength over the investigated range rather
than showing a stable alternative optimum.

Therefore, α = 1.0 is retained as the final Ridge configuration for model
comparison. This provides a simple, reproducible regularized linear baseline
without introducing additional complexity that does not produce a meaningful
predictive improvement.

The final Ridge model remains embedded in a StandardScaler → Ridge pipeline,
with scaling parameters fitted only on the training data of each chronological
fold.

In [41]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

final_ridge_alpha = 1.0

final_ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(
        alpha=final_ridge_alpha,
        fit_intercept=True,
        solver="auto"
    ))
])

print("Final Ridge Configuration")
print("=========================")
print(f"Alpha: {final_ridge_alpha}")
print("Preprocessing: StandardScaler")
print("Model: Ridge Regression")
print("Decision: Baseline α = 1.0 retained")
print("Reason: Nested tuning produced negligible practical improvement")

Final Ridge Configuration
Alpha: 1.0
Preprocessing: StandardScaler
Model: Ridge Regression
Decision: Baseline α = 1.0 retained
Reason: Nested tuning produced negligible practical improvement


In [ ]:
## F1.14 — XGBoost Model Development

### F1.14.1 — Modelling Objective

Ridge Regression established a strong regularized linear baseline, achieving
a mean chronological validation RMSE of approximately 0.0171 V and mean
R² of approximately 0.97.

The next stage evaluates whether a nonlinear model can provide a meaningful
and consistent improvement over this baseline.

XGBoost is selected because tree-based boosting can represent nonlinear
relationships and interactions between PEMFC operating variables without
requiring those relationships to be specified manually.

The XGBoost model developed in this section is distinct from the XGBoost
estimator previously used within Boruta feature selection. The Boruta
configuration was selected for stable feature-importance estimation, whereas
the present model will be developed specifically for voltage-prediction
performance.

The same 20 retained predictors and the same four expanding chronological
validation folds are used so that XGBoost can be compared directly and fairly
with Ridge Regression.

In [ ]:
### F1.14.2 — Baseline Predictive XGBoost Configuration

A baseline XGBoost regression model is first established before
hyperparameter tuning.

The purpose of this baseline is to determine how well a nonlinear
tree-boosting model can predict voltage under the same chronological
validation structure used for Ridge Regression.

This predictive XGBoost model is implemented using XGBRegressor because
voltage is a continuous regression target.

The initial configuration is intentionally reasonable but untuned. It is
not assumed to be optimal. Hyperparameter optimisation will be performed
separately using training-only chronological validation.

Unlike Ridge Regression, XGBoost does not require predictor standardisation
because tree-based models split observations according to feature values
rather than relying on coefficient magnitudes.

The same 20 retained predictors and four expanding chronological folds are
used to ensure a fair comparison with Ridge.

In [42]:
from xgboost import XGBRegressor

xgb_baseline_config = {
    "objective": "reg:squarederror",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0,
    "reg_alpha": 0,
    "reg_lambda": 1,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0
}

xgb_baseline_model = XGBRegressor(**xgb_baseline_config)

print("Baseline Predictive XGBoost Configuration")
print("=========================================")

for parameter, value in xgb_baseline_config.items():
    print(f"{parameter}: {value}")

Baseline Predictive XGBoost Configuration
objective: reg:squarederror
n_estimators: 300
learning_rate: 0.05
max_depth: 6
min_child_weight: 1
subsample: 0.8
colsample_bytree: 0.8
gamma: 0
reg_alpha: 0
reg_lambda: 1
tree_method: hist
random_state: 42
n_jobs: -1
verbosity: 0


In [ ]:
### F1.14.3 — Chronological XGBoost Evaluation Function

A common evaluation function is defined to train and evaluate the baseline
XGBRegressor within each chronological fold.

For every fold, the model is trained only on the earlier durability stages
and evaluated on the subsequent unseen validation stages.

The same 20 retained predictors and the same RMSE, MAE and R² metrics used
for Ridge Regression are retained. This ensures that differences between
Ridge and XGBoost reflect model behaviour rather than differences in the
evaluation procedure.

In [43]:
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_xgb_fold(
    X_train,
    y_train,
    X_validation,
    y_validation,
    feature_set,
    fold_name
):
    # Select the same retained predictors
    X_train_selected = X_train[feature_set]
    X_validation_selected = X_validation[feature_set]

    # Fresh copy of the baseline XGBoost model
    model = clone(xgb_baseline_model)

    # Train only on the chronological training region
    model.fit(X_train_selected, y_train)

    # Predict the later validation stages
    y_pred = model.predict(X_validation_selected)

    # Evaluation metrics
    rmse = np.sqrt(
        mean_squared_error(y_validation, y_pred)
    )

    mae = mean_absolute_error(
        y_validation, y_pred
    )

    r2 = r2_score(
        y_validation, y_pred
    )

    results = {
        "fold": fold_name,
        "n_features": len(feature_set),
        "train_observations": len(X_train),
        "validation_observations": len(X_validation),
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

    return results, model

In [ ]:
### F1.14.4 — Baseline XGBoost Across Chronological Folds

The baseline XGBRegressor is evaluated across the same four expanding
chronological folds previously used for Ridge Regression.

For each fold, the model is trained on the earlier durability-stage blocks
and tested on the immediately subsequent unseen stages.

Using identical predictors, chronological splits and evaluation metrics allows
the baseline XGBoost results to be compared directly with the Ridge baseline.

This stage answers whether nonlinear tree-based modelling provides an immediate
predictive advantage over the regularized linear baseline before any XGBoost
hyperparameter tuning is performed.

In [46]:
import pandas as pd

xgb_baseline_results = []
xgb_baseline_models = {}

for fold_name, fold_info in boruta_outer_folds.items():

    train_labels = fold_info["train_labels"]
    validation_labels = fold_info["validation_labels"]

    # Chronological train/validation split
    train_mask = development_df["operating_hour"].isin(train_labels)
    validation_mask = development_df["operating_hour"].isin(validation_labels)

    X_train = development_df.loc[
        train_mask, boruta_candidate_predictors
    ]
    y_train = development_df.loc[
        train_mask, "voltage"
    ]

    X_validation = development_df.loc[
        validation_mask, boruta_candidate_predictors
    ]
    y_validation = development_df.loc[
        validation_mask, "voltage"
    ]

    # Evaluate baseline XGBoost
    fold_results, fitted_model = evaluate_xgb_fold(
        X_train=X_train,
        y_train=y_train,
        X_validation=X_validation,
        y_validation=y_validation,
        feature_set=boruta_candidate_predictors,
        fold_name=fold_name
    )

    xgb_baseline_results.append(fold_results)
    xgb_baseline_models[fold_name] = fitted_model

    print(
        f"{fold_name} | "
        f"Train: {train_labels[0]}–{train_labels[-1]} h | "
        f"Validation: {validation_labels} | "
        f"RMSE: {fold_results['rmse']:.6f} V | "
        f"MAE: {fold_results['mae']:.6f} V | "
        f"R²: {fold_results['r2']:.6f}"
    )

# Convert results to table
xgb_baseline_results_df = pd.DataFrame(xgb_baseline_results)

print("\nBaseline XGBoost Results")
print("========================")
display(xgb_baseline_results_df)

Fold_1 | Train: 50–450 h | Validation: [500, 550] | RMSE: 0.007211 V | MAE: 0.005092 V | R²: 0.994562
Fold_2 | Train: 50–550 h | Validation: [600, 650] | RMSE: 0.009716 V | MAE: 0.006591 V | R²: 0.990547
Fold_3 | Train: 50–650 h | Validation: [700, 750] | RMSE: 0.011420 V | MAE: 0.009267 V | R²: 0.987520
Fold_4 | Train: 50–750 h | Validation: [800, 850] | RMSE: 0.010205 V | MAE: 0.007668 V | R²: 0.989318

Baseline XGBoost Results


,fold,n_features,train_observations,validation_observations,rmse,mae,r2
0,Fold_1,20,1614240,358720,0.007211,0.005092,0.994562
1,Fold_2,20,1972960,358720,0.009716,0.006591,0.990547
2,Fold_3,20,2331680,358720,0.011420,0.009267,0.987520
3,Fold_4,20,2690400,358720,0.010205,0.007668,0.989318


In [ ]:
### F1.14.5 — Baseline XGBoost Performance Summary and Ridge Comparison

The baseline XGBRegressor was evaluated across the same four expanding
chronological folds used for Ridge Regression.

XGBoost achieved consistently strong later-stage validation performance,
with R² remaining above 0.987 across all four folds.

Mean baseline XGBoost performance was approximately:

- RMSE = 0.00964 V
- MAE = 0.00715 V
- R² = 0.99049

For comparison, the Ridge baseline achieved:

- RMSE = 0.01710 V
- MAE = 0.01367 V
- R² = 0.97020

Therefore, baseline XGBoost reduced mean RMSE by approximately 44% and
mean MAE by approximately 48% relative to Ridge.

The improvement is sufficiently large to indicate that nonlinear relationships
and/or interactions between PEMFC operating variables contain important
predictive information that cannot be fully represented by the regularized
linear Ridge model.

Fold 3 remained the most difficult validation period for both Ridge and
XGBoost. This suggests that the 700–750 h region may represent a more
challenging operating or distributional regime, although this should not be
attributed directly to degradation without further evidence.

In [47]:
# Mean baseline XGBoost performance

xgb_mean_rmse = xgb_baseline_results_df["rmse"].mean()
xgb_mean_mae = xgb_baseline_results_df["mae"].mean()
xgb_mean_r2 = xgb_baseline_results_df["r2"].mean()

# Ridge baseline means already obtained previously
ridge_mean_rmse = ridge_chronological_results["rmse"].mean()
ridge_mean_mae = ridge_chronological_results["mae"].mean()
ridge_mean_r2 = ridge_chronological_results["r2"].mean()

# Relative improvements
rmse_reduction_pct = (
    (ridge_mean_rmse - xgb_mean_rmse) / ridge_mean_rmse
) * 100

mae_reduction_pct = (
    (ridge_mean_mae - xgb_mean_mae) / ridge_mean_mae
) * 100

r2_improvement = xgb_mean_r2 - ridge_mean_r2

print("Baseline Ridge vs Baseline XGBoost")
print("==================================")

print("\nRidge")
print(f"Mean RMSE: {ridge_mean_rmse:.6f} V")
print(f"Mean MAE:  {ridge_mean_mae:.6f} V")
print(f"Mean R²:   {ridge_mean_r2:.6f}")

print("\nXGBoost")
print(f"Mean RMSE: {xgb_mean_rmse:.6f} V")
print(f"Mean MAE:  {xgb_mean_mae:.6f} V")
print(f"Mean R²:   {xgb_mean_r2:.6f}")

print("\nImprovement of XGBoost over Ridge")
print(f"RMSE reduction: {rmse_reduction_pct:.2f}%")
print(f"MAE reduction:  {mae_reduction_pct:.2f}%")
print(f"R² increase:    {r2_improvement:.6f}")

Baseline Ridge vs Baseline XGBoost

Ridge
Mean RMSE: 0.017102 V
Mean MAE:  0.013666 V
Mean R²:   0.970199

XGBoost
Mean RMSE: 0.009638 V
Mean MAE:  0.007154 V
Mean R²:   0.990487

Improvement of XGBoost over Ridge
RMSE reduction: 43.64%
MAE reduction:  47.65%
R² increase:    0.020288


In [ ]:
### F1.14.6 — XGBoost Hyperparameter Tuning Strategy

The baseline XGBRegressor demonstrated strong chronological validation
performance, but its initial hyperparameters were not optimised specifically
for the voltage-prediction task.

A systematic hyperparameter optimisation procedure is therefore used to
investigate whether predictive performance and later-stage generalisation can
be improved.

Because XGBoost contains several interacting hyperparameters controlling
boosting rate, tree complexity, sampling and regularisation, a randomized
search is preferred over a small manually selected set of model
configurations.

The search considers the number of boosting trees, learning rate, maximum
tree depth, minimum child weight, row and feature subsampling, split
regularisation, and L1/L2 regularisation.

Hyperparameter selection is embedded within the chronological validation
framework. Within each outer-training region, candidate configurations are
evaluated using the two most recent valid inner chronological validation
periods.

Using more than one inner validation period reduces dependence on a single
durability interval, while focusing the tuning procedure on later-stage
generalisation.

Performance across the two inner validation periods is aggregated using
mean RMSE, and the configuration with the lowest mean inner-validation RMSE
is selected for the corresponding outer fold.

The selected configuration is subsequently refitted using the complete
outer-training region and evaluated once on the untouched outer-validation
stages.

This nested chronological procedure separates hyperparameter optimisation
from outer model evaluation.

The final 900–1000 h holdout remains completely excluded from feature
selection, hyperparameter optimisation and model selection.

In [ ]:
### F1.14.7 — XGBoost Hyperparameter Search Space

A broad hyperparameter search space is defined for the predictive XGBoost
model.

The search covers parameters controlling boosting rate, ensemble size, tree
complexity, observation and feature subsampling, split conservativeness, and
L1/L2 regularisation.

Continuous parameters are sampled from probability distributions rather than
restricted to a small number of manually selected values. This provides
broader coverage of the hyperparameter space while avoiding the rigid
Cartesian structure of an exhaustive grid.

Log-uniform distributions are used for parameters whose plausible values span
multiple orders of magnitude, particularly learning rate and regularisation
strength. Uniform distributions are used for sampling proportions, while
integer distributions are used for tree number, depth and minimum child
weight.

The search space is intentionally broad because hyperparameter selection will
subsequently be based on chronological inner-validation performance rather
than on assumptions about which configuration should perform best.

In [48]:
from scipy.stats import randint, uniform, loguniform

# Formal XGBoost hyperparameter search space
xgb_param_distributions = {

    # Boosting process
    "n_estimators": randint(200, 1201),
    "learning_rate": loguniform(0.01, 0.20),

    # Tree complexity
    "max_depth": randint(3, 11),
    "min_child_weight": randint(1, 16),

    # Stochastic sampling
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),

    # Split regularisation
    "gamma": uniform(0.0, 1.0),

    # L1 and L2 regularisation
    "reg_alpha": loguniform(1e-4, 10.0),
    "reg_lambda": loguniform(0.1, 20.0)
}

print("XGBoost Hyperparameter Search Space")
print("===================================")

print("n_estimators:       integer 200–1200")
print("learning_rate:      log-uniform 0.01–0.20")
print("max_depth:          integer 3–10")
print("min_child_weight:   integer 1–15")
print("subsample:          uniform 0.60–1.00")
print("colsample_bytree:   uniform 0.60–1.00")
print("gamma:              uniform 0.00–1.00")
print("reg_alpha:          log-uniform 0.0001–10")
print("reg_lambda:         log-uniform 0.1–20")

XGBoost Hyperparameter Search Space
n_estimators:       integer 200–1200
learning_rate:      log-uniform 0.01–0.20
max_depth:          integer 3–10
min_child_weight:   integer 1–15
subsample:          uniform 0.60–1.00
colsample_bytree:   uniform 0.60–1.00
gamma:              uniform 0.00–1.00
reg_alpha:          log-uniform 0.0001–10
reg_lambda:         log-uniform 0.1–20


In [ ]:
### F1.14.8 — Nested Inner Chronological Validation Design

Expanding inner chronological validation splits are first constructed within
each outer-training region.

Each inner split trains on earlier durability stages and validates on later
stages. No observation belonging to the corresponding outer-validation
period is used during hyperparameter selection.

The complete set of valid inner splits is retained for transparency and to
verify the chronological structure.

For the formal XGBoost tuning procedure, the two most recent valid inner
validation periods available within each outer fold are used.

This provides repeated chronological validation while giving greater emphasis
to the later durability regions that are most relevant to the study's
later-stage generalisation objective.

The final 900–1000 h holdout remains completely outside both inner and outer
model-development procedures.

In [49]:
# Construct expanding inner chronological splits for each outer fold

xgb_inner_folds = {}

for outer_fold_name, outer_fold_info in boruta_outer_folds.items():

    outer_train_labels = outer_fold_info["train_labels"]

    inner_splits = []

    # Use two consecutive durability stages for validation
    # and require substantial earlier history for training.
    for validation_start_idx in range(5, len(outer_train_labels) - 1, 2):

        inner_train_labels = outer_train_labels[:validation_start_idx]

        inner_validation_labels = outer_train_labels[
            validation_start_idx:validation_start_idx + 2
        ]

        if len(inner_validation_labels) == 2:

            inner_splits.append({
                "train_labels": inner_train_labels,
                "validation_labels": inner_validation_labels
            })

    xgb_inner_folds[outer_fold_name] = inner_splits


# Display the resulting structure
print("Nested Inner Chronological Validation Structure")
print("================================================")

for outer_fold_name, inner_splits in xgb_inner_folds.items():

    print(f"\n{outer_fold_name}")
    print("-" * len(outer_fold_name))

    for i, split in enumerate(inner_splits, start=1):

        train_labels = split["train_labels"]
        validation_labels = split["validation_labels"]

        print(
            f"Inner_{i}: "
            f"Train {train_labels[0]}–{train_labels[-1]} h | "
            f"Validation {validation_labels}"
        )

Nested Inner Chronological Validation Structure

Fold_1
------
Inner_1: Train 50–250 h | Validation [300, 350]
Inner_2: Train 50–350 h | Validation [400, 450]

Fold_2
------
Inner_1: Train 50–250 h | Validation [300, 350]
Inner_2: Train 50–350 h | Validation [400, 450]
Inner_3: Train 50–450 h | Validation [500, 550]

Fold_3
------
Inner_1: Train 50–250 h | Validation [300, 350]
Inner_2: Train 50–350 h | Validation [400, 450]
Inner_3: Train 50–450 h | Validation [500, 550]
Inner_4: Train 50–550 h | Validation [600, 650]

Fold_4
------
Inner_1: Train 50–250 h | Validation [300, 350]
Inner_2: Train 50–350 h | Validation [400, 450]
Inner_3: Train 50–450 h | Validation [500, 550]
Inner_4: Train 50–550 h | Validation [600, 650]
Inner_5: Train 50–650 h | Validation [700, 750]


In [53]:
# Retain the two most recent valid inner splits for formal XGBoost tuning

xgb_tuning_inner_folds = {
    outer_fold: inner_splits[-2:]
    for outer_fold, inner_splits in xgb_inner_folds.items()
}

print("\nInner Splits Retained for Formal XGBoost Tuning")
print("================================================")

for outer_fold_name, inner_splits in xgb_tuning_inner_folds.items():

    print(f"\n{outer_fold_name}")

    for i, split in enumerate(inner_splits, start=1):

        train_labels = split["train_labels"]
        validation_labels = split["validation_labels"]

        print(
            f"Tuning Inner_{i}: "
            f"Train {train_labels[0]}–{train_labels[-1]} h | "
            f"Validation {validation_labels}"
        )


Inner Splits Retained for Formal XGBoost Tuning

Fold_1
Tuning Inner_1: Train 50–250 h | Validation [300, 350]
Tuning Inner_2: Train 50–350 h | Validation [400, 450]

Fold_2
Tuning Inner_1: Train 50–350 h | Validation [400, 450]
Tuning Inner_2: Train 50–450 h | Validation [500, 550]

Fold_3
Tuning Inner_1: Train 50–450 h | Validation [500, 550]
Tuning Inner_2: Train 50–550 h | Validation [600, 650]

Fold_4
Tuning Inner_1: Train 50–550 h | Validation [600, 650]
Tuning Inner_2: Train 50–650 h | Validation [700, 750]


In [ ]:
### F1.14.9 — Generation of XGBoost Hyperparameter Candidates

Thirty XGBoost configurations are sampled reproducibly from the previously
defined broad hyperparameter distributions.

The same sampled configurations are evaluated within every outer fold. This
ensures that differences in selected hyperparameters across outer folds arise
from differences in chronological training history rather than from different
random search samples.

The number of configurations was selected to provide meaningful exploration
of the multidimensional XGBoost search space while maintaining a
proportionate model-development procedure for the large high-frequency
durability dataset.

Combined with two chronological inner validation periods per outer fold, the
procedure requires 240 inner model fits in total.

A fixed random seed is used so that the sampled configurations are fully
reproducible.

At this stage, candidate configurations are generated only; no models are
trained.

In [54]:
from sklearn.model_selection import ParameterSampler
import pandas as pd

# Number of candidate configurations
n_xgb_candidates = 30

# Reproducible sampling from the formal search space
xgb_sampled_candidates = list(
    ParameterSampler(
        param_distributions=xgb_param_distributions,
        n_iter=n_xgb_candidates,
        random_state=42
    )
)

print("XGBoost Hyperparameter Candidate Generation")
print("============================================")
print(f"Number of sampled configurations: {len(xgb_sampled_candidates)}")

# Preview the first five configurations
xgb_candidate_preview = pd.DataFrame(
    xgb_sampled_candidates[:5]
)

print("\nFirst 5 sampled configurations:")
display(xgb_candidate_preview)

# Planned number of inner model fits
total_planned_fits = (
    len(xgb_sampled_candidates)
    * sum(len(splits) for splits in xgb_tuning_inner_folds.values())
)

print(f"\nTotal planned inner XGBoost fits: {total_planned_fits}")

XGBoost Hyperparameter Candidate Generation
Number of sampled configurations: 30

First 5 sampled configurations:


,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample
0,0.749816,0.950714,0.089608,7,5,814,0.016950,0.169842,0.783700
1,0.733483,0.142867,0.070279,7,2,543,1.452825,0.308034,0.672730
2,0.673362,0.304242,0.048164,6,9,760,0.042060,0.831940,0.618666
3,0.989502,0.232771,0.013118,8,3,1075,0.037254,2.307617,0.618580
4,0.843018,0.170524,0.012152,6,9,515,0.065530,0.770646,0.606387



Total planned inner XGBoost fits: 240


In [ ]:
### F1.14.10 — Focused Nested Chronological XGBoost Hyperparameter Search

The 30 sampled XGBoost configurations are evaluated using the two most recent
valid inner chronological validation periods available within each outer fold.

For each candidate configuration, the model is trained separately on both
inner-training regions and evaluated on their corresponding later validation
periods.

RMSE is calculated for each inner validation period. The mean inner-validation
RMSE is used as the primary tuning criterion, while the standard deviation is
retained to indicate variation in performance between the two chronological
validation periods.

The configuration with the lowest mean inner-validation RMSE is selected
independently within each outer fold.

The corresponding outer-validation period is not used during hyperparameter
selection and therefore remains available for an unbiased evaluation of the
tuned XGBoost procedure.

The final 900–1000 h holdout remains completely untouched.

In [55]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import time
from pathlib import Path

# Store all tuning results
xgb_nested_tuning_results = []

# Save checkpoints locally
checkpoint_dir = Path("../results/xgb_tuning_checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Focused Nested Chronological XGBoost Search")
print("============================================")
print(f"Candidate configurations per outer fold: {len(xgb_sampled_candidates)}")
print(
    f"Total planned fits: "
    f"{len(xgb_sampled_candidates) * sum(len(v) for v in xgb_tuning_inner_folds.values())}"
)

overall_start = time.time()

for outer_fold_name, inner_splits in xgb_tuning_inner_folds.items():

    print(f"\n{'=' * 60}")
    print(f"Starting {outer_fold_name}")
    print(f"Inner splits: {len(inner_splits)}")
    print(f"{'=' * 60}")

    fold_start = time.time()
    fold_candidate_results = []

    for candidate_idx, candidate_params in enumerate(
        xgb_sampled_candidates,
        start=1
    ):

        inner_rmse_scores = []

        for inner_idx, inner_split in enumerate(inner_splits, start=1):

            inner_train_labels = inner_split["train_labels"]
            inner_validation_labels = inner_split["validation_labels"]

            # Chronological inner datasets
            inner_train_mask = development_df["operating_hour"].isin(
                inner_train_labels
            )

            inner_validation_mask = development_df["operating_hour"].isin(
                inner_validation_labels
            )

            X_inner_train = development_df.loc[
                inner_train_mask,
                boruta_candidate_predictors
            ]

            y_inner_train = development_df.loc[
                inner_train_mask,
                "voltage"
            ]

            X_inner_validation = development_df.loc[
                inner_validation_mask,
                boruta_candidate_predictors
            ]

            y_inner_validation = development_df.loc[
                inner_validation_mask,
                "voltage"
            ]

            # Candidate XGBoost model
            model = XGBRegressor(
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=-1,
                verbosity=0,
                **candidate_params
            )

            model.fit(
                X_inner_train,
                y_inner_train
            )

            y_inner_pred = model.predict(
                X_inner_validation
            )

            inner_rmse = np.sqrt(
                mean_squared_error(
                    y_inner_validation,
                    y_inner_pred
                )
            )

            inner_rmse_scores.append(inner_rmse)

        # Aggregate inner-fold performance
        candidate_result = {
            "outer_fold": outer_fold_name,
            "candidate": candidate_idx,
            "mean_inner_rmse": np.mean(inner_rmse_scores),
            "std_inner_rmse": np.std(inner_rmse_scores),
            "min_inner_rmse": np.min(inner_rmse_scores),
            "max_inner_rmse": np.max(inner_rmse_scores),
            **candidate_params
        }

        for inner_idx, score in enumerate(inner_rmse_scores, start=1):
            candidate_result[f"inner_{inner_idx}_rmse"] = score

        fold_candidate_results.append(candidate_result)
        xgb_nested_tuning_results.append(candidate_result)

        # Progress output every 5 candidates
        if candidate_idx % 5 == 0 or candidate_idx == 1:

            elapsed = time.time() - fold_start

            best_so_far = min(
                result["mean_inner_rmse"]
                for result in fold_candidate_results
            )

            print(
                f"{outer_fold_name} | "
                f"Candidate {candidate_idx:>2}/{len(xgb_sampled_candidates)} | "
                f"Current RMSE: {candidate_result['mean_inner_rmse']:.6f} V | "
                f"Best so far: {best_so_far:.6f} V | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

        # Checkpoint every 10 candidates
        if candidate_idx % 10 == 0:

            checkpoint_df = pd.DataFrame(
                fold_candidate_results
            )

            checkpoint_df.to_csv(
                checkpoint_dir /
                f"{outer_fold_name}_checkpoint.csv",
                index=False
            )

    # Complete outer-fold results
    fold_results_df = pd.DataFrame(
        fold_candidate_results
    )

    fold_results_df.to_csv(
        checkpoint_dir /
        f"{outer_fold_name}_complete.csv",
        index=False
    )

    # Best candidate
    best_row = fold_results_df.loc[
        fold_results_df["mean_inner_rmse"].idxmin()
    ]

    print(f"\nBest configuration for {outer_fold_name}")
    print(f"Candidate: {int(best_row['candidate'])}")
    print(
        f"Mean inner RMSE: "
        f"{best_row['mean_inner_rmse']:.6f} V"
    )
    print(
        f"Inner RMSE SD: "
        f"{best_row['std_inner_rmse']:.6f} V"
    )

    fold_elapsed = time.time() - fold_start
    print(
        f"{outer_fold_name} completed in "
        f"{fold_elapsed / 60:.1f} minutes"
    )


# Combined tuning results
xgb_nested_tuning_results_df = pd.DataFrame(
    xgb_nested_tuning_results
)

overall_elapsed = time.time() - overall_start

print("\nFocused XGBoost tuning completed.")
print(
    f"Candidate evaluations: "
    f"{len(xgb_nested_tuning_results_df)}"
)
print(
    f"Total runtime: "
    f"{overall_elapsed / 60:.1f} minutes"
)

Focused Nested Chronological XGBoost Search
Candidate configurations per outer fold: 30
Total planned fits: 240

Starting Fold_1
Inner splits: 2
Fold_1 | Candidate  1/30 | Current RMSE: 0.017801 V | Best so far: 0.017801 V | Elapsed: 2.0 min
Fold_1 | Candidate  5/30 | Current RMSE: 0.016017 V | Best so far: 0.015985 V | Elapsed: 10.0 min
Fold_1 | Candidate 10/30 | Current RMSE: 0.019999 V | Best so far: 0.015985 V | Elapsed: 20.0 min
Fold_1 | Candidate 15/30 | Current RMSE: 0.017483 V | Best so far: 0.015985 V | Elapsed: 26.0 min
Fold_1 | Candidate 20/30 | Current RMSE: 0.016110 V | Best so far: 0.015985 V | Elapsed: 31.3 min
Fold_1 | Candidate 25/30 | Current RMSE: 0.017998 V | Best so far: 0.015985 V | Elapsed: 39.8 min
Fold_1 | Candidate 30/30 | Current RMSE: 0.017541 V | Best so far: 0.015954 V | Elapsed: 45.1 min

Best configuration for Fold_1
Candidate: 26
Mean inner RMSE: 0.015954 V
Inner RMSE SD: 0.006376 V
Fold_1 completed in 45.1 minutes

Starting Fold_2
Inner splits: 2
Fold_

In [58]:
# Extract all XGBoost configurations selected during nested tuning

selected_candidate_numbers = [25, 26, 28]

selected_xgb_configs = {}

for candidate_number in selected_candidate_numbers:
    
    # Python indexing starts from 0
    params = xgb_sampled_candidates[candidate_number - 1].copy()
    
    selected_xgb_configs[candidate_number] = params


# Display configurations in a comparison table
selected_xgb_configs_df = pd.DataFrame({
    f"Candidate_{candidate_number}": params
    for candidate_number, params in selected_xgb_configs.items()
})

print("Selected XGBoost Hyperparameter Configurations")
print("==============================================")
print("Candidate 25 → Selected for Folds 3 and 4")
print("Candidate 26 → Selected for Fold 1")
print("Candidate 28 → Selected for Fold 2")

display(selected_xgb_configs_df)

Selected XGBoost Hyperparameter Configurations
Candidate 25 → Selected for Folds 3 and 4
Candidate 26 → Selected for Fold 1
Candidate 28 → Selected for Fold 2


,Candidate_25,Candidate_26,Candidate_28
colsample_bytree,0.736427,0.806654,0.914136
gamma,0.113474,0.260829,0.668988
learning_rate,0.159608,0.197768,0.056950
max_depth,8.000000,8.000000,10.000000
min_child_weight,10.000000,12.000000,14.000000
n_estimators,810.000000,652.000000,672.000000
reg_alpha,0.001389,3.177535,0.207735
reg_lambda,0.252683,2.862763,0.102718
subsample,0.992867,0.735612,0.664323


In [ ]:
### F1.14.12 — Tuned XGBoost Outer-Fold Evaluation

The XGBoost configuration selected within each outer fold is now refitted on
the complete outer-training region and evaluated on the corresponding
untouched outer-validation stages.

This provides an unbiased assessment of whether the hyperparameter tuning
procedure improves later-stage generalisation relative to the baseline
XGBoost model.

The selected configurations are:

- Fold 1 → Candidate 26
- Fold 2 → Candidate 28
- Fold 3 → Candidate 25
- Fold 4 → Candidate 25

The final 900–1000 h holdout remains completely excluded from this stage.

In [59]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import time

# Map each outer fold to the candidate selected during nested tuning
fold_selected_candidates = {
    "Fold_1": 26,
    "Fold_2": 28,
    "Fold_3": 25,
    "Fold_4": 25
}

# Outer chronological fold definitions
outer_fold_definitions = {
    "Fold_1": {
        "train_hours": list(range(50, 451, 50)),
        "validation_hours": [500, 550]
    },
    "Fold_2": {
        "train_hours": list(range(50, 551, 50)),
        "validation_hours": [600, 650]
    },
    "Fold_3": {
        "train_hours": list(range(50, 651, 50)),
        "validation_hours": [700, 750]
    },
    "Fold_4": {
        "train_hours": list(range(50, 751, 50)),
        "validation_hours": [800, 850]
    }
}

tuned_xgb_outer_results = []
tuned_xgb_outer_models = {}

print("Tuned XGBoost Outer-Fold Evaluation")
print("====================================")

for fold_name, fold_info in outer_fold_definitions.items():

    start_time = time.time()

    candidate_number = fold_selected_candidates[fold_name]
    candidate_params = selected_xgb_configs[candidate_number]

    train_mask = development_df["operating_hour"].isin(
        fold_info["train_hours"]
    )

    validation_mask = development_df["operating_hour"].isin(
        fold_info["validation_hours"]
    )

    X_train = development_df.loc[
        train_mask, boruta_candidate_predictors
    ]

    y_train = development_df.loc[
        train_mask, "voltage"
    ]

    X_validation = development_df.loc[
        validation_mask, boruta_candidate_predictors
    ]

    y_validation = development_df.loc[
        validation_mask, "voltage"
    ]

    model = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
        **candidate_params
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_validation)

    rmse = np.sqrt(
        mean_squared_error(y_validation, y_pred)
    )

    mae = mean_absolute_error(
        y_validation, y_pred
    )

    r2 = r2_score(
        y_validation, y_pred
    )

    runtime = time.time() - start_time

    tuned_xgb_outer_results.append({
        "Fold": fold_name,
        "Candidate": candidate_number,
        "Train_End_h": max(fold_info["train_hours"]),
        "Validation_Hours": str(fold_info["validation_hours"]),
        "Train_Observations": len(X_train),
        "Validation_Observations": len(X_validation),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Runtime_s": runtime
    })

    tuned_xgb_outer_models[fold_name] = model

    print(
        f"{fold_name} | "
        f"Candidate {candidate_number} | "
        f"Train ≤ {max(fold_info['train_hours'])} h | "
        f"Validation {fold_info['validation_hours']} | "
        f"RMSE: {rmse:.6f} V | "
        f"MAE: {mae:.6f} V | "
        f"R²: {r2:.6f} | "
        f"Runtime: {runtime:.1f} s"
    )


tuned_xgb_outer_results_df = pd.DataFrame(
    tuned_xgb_outer_results
)

print("\nTuned XGBoost Outer-Fold Results")
print("================================")

display(tuned_xgb_outer_results_df)

print("\nMean Tuned XGBoost Performance")
print("==============================")
print(
    f"Mean RMSE: "
    f"{tuned_xgb_outer_results_df['RMSE'].mean():.6f} V"
)
print(
    f"Mean MAE:  "
    f"{tuned_xgb_outer_results_df['MAE'].mean():.6f} V"
)
print(
    f"Mean R²:   "
    f"{tuned_xgb_outer_results_df['R2'].mean():.6f}"
)

Tuned XGBoost Outer-Fold Evaluation
Fold_1 | Candidate 26 | Train ≤ 450 h | Validation [500, 550] | RMSE: 0.008462 V | MAE: 0.005802 V | R²: 0.992511 | Runtime: 45.5 s
Fold_2 | Candidate 28 | Train ≤ 550 h | Validation [600, 650] | RMSE: 0.011145 V | MAE: 0.007786 V | R²: 0.987562 | Runtime: 63.6 s
Fold_3 | Candidate 25 | Train ≤ 650 h | Validation [700, 750] | RMSE: 0.012460 V | MAE: 0.010124 V | R²: 0.985145 | Runtime: 69.8 s
Fold_4 | Candidate 25 | Train ≤ 750 h | Validation [800, 850] | RMSE: 0.009838 V | MAE: 0.007026 V | R²: 0.990072 | Runtime: 82.3 s

Tuned XGBoost Outer-Fold Results


,Fold,Candidate,Train_End_h,Validation_Hours,Train_Observations,Validation_Observations,RMSE,MAE,R2,Runtime_s
0,Fold_1,26,450,"[500, 550]",1614240,358720,0.008462,0.005802,0.992511,45.508696
1,Fold_2,28,550,"[600, 650]",1972960,358720,0.011145,0.007786,0.987562,63.560033
2,Fold_3,25,650,"[700, 750]",2331680,358720,0.012460,0.010124,0.985145,69.750472
3,Fold_4,25,750,"[800, 850]",2690400,358720,0.009838,0.007026,0.990072,82.300938



Mean Tuned XGBoost Performance
Mean RMSE: 0.010476 V
Mean MAE:  0.007684 V
Mean R²:   0.988823


In [ ]:
### F1.14.13 — XGBoost Hyperparameter Tuning Decision

Nested chronological hyperparameter tuning did not provide a consistent
improvement over the baseline XGBoost configuration.

The baseline XGBoost model achieved mean outer-fold performance of
approximately RMSE = 0.00964 V, MAE = 0.00715 V and R² = 0.99049.

The fold-specific tuned configurations achieved mean performance of
RMSE = 0.01048 V, MAE = 0.00768 V and R² = 0.98882.

The tuned configuration improved RMSE only in Fold 4, while baseline XGBoost
performed better in Folds 1, 2 and 3. Overall, tuning increased mean RMSE by
approximately 8.7%.

Therefore, the improvements observed during inner hyperparameter selection
did not consistently transfer to unseen later durability stages.

The baseline XGBoost configuration is consequently retained for final model
comparison because it demonstrated better and more consistent chronological
generalisation while also using a simpler computational configuration.

The final 900–1000 h holdout remains untouched and has not contributed to
this decision.

In [60]:
# Extract and display the baseline XGBoost configuration
# retained after chronological hyperparameter evaluation

baseline_xgb_config_df = pd.DataFrame(
    list(xgb_baseline_config.items()),
    columns=["Hyperparameter", "Value"]
)

print("Baseline XGBoost Configuration")
print("==============================")
print("Decision: Retained after nested chronological tuning")
print("Reason: Better mean outer-fold generalisation than tuned configurations")

display(baseline_xgb_config_df)

Baseline XGBoost Configuration
Decision: Retained after nested chronological tuning
Reason: Better mean outer-fold generalisation than tuned configurations


,Hyperparameter,Value
0,objective,reg:squarederror
1,n_estimators,300
2,learning_rate,0.05
3,max_depth,6
4,min_child_weight,1
5,subsample,0.8
6,colsample_bytree,0.8
7,gamma,0
8,reg_alpha,0
9,reg_lambda,1


In [ ]:
## F1.15 — Final Chronological Model Comparison

Ridge Regression and XGBoost are now compared using the same four expanding
chronological validation folds.

For Ridge Regression, α = 1.0 was retained because nested tuning produced only
negligible changes in predictive performance.

For XGBoost, the baseline configuration was retained because nested
hyperparameter tuning did not consistently improve unseen later-stage
generalisation and produced a higher mean outer-fold RMSE.

The comparison therefore evaluates the final selected configuration of each
model under an identical chronological validation framework using RMSE, MAE
and R².

The final 900–1000 h holdout remains completely excluded from this comparison.

In [61]:
import pandas as pd

# Final Ridge chronological validation results
ridge_final_comparison = pd.DataFrame({
    "Fold": ["Fold_1", "Fold_2", "Fold_3", "Fold_4"],
    "Ridge_RMSE": [0.018224, 0.016332, 0.019800, 0.014051],
    "Ridge_MAE":  [0.014882, 0.012935, 0.016798, 0.010049],
    "Ridge_R2":   [0.965270, 0.973288, 0.962488, 0.979747]
})

# Baseline XGBoost chronological validation results
xgb_final_comparison = pd.DataFrame({
    "Fold": ["Fold_1", "Fold_2", "Fold_3", "Fold_4"],
    "XGB_RMSE": [0.007211, 0.009716, 0.011420, 0.010205],
    "XGB_MAE":  [0.005092, 0.006591, 0.009267, 0.007668],
    "XGB_R2":   [0.994562, 0.990547, 0.987520, 0.989318]
})

# Merge both models into one comparison table
final_model_comparison_df = ridge_final_comparison.merge(
    xgb_final_comparison,
    on="Fold"
)

# Calculate fold-wise RMSE improvement of XGBoost over Ridge
final_model_comparison_df["RMSE_Reduction_%"] = (
    (
        final_model_comparison_df["Ridge_RMSE"]
        - final_model_comparison_df["XGB_RMSE"]
    )
    / final_model_comparison_df["Ridge_RMSE"]
    * 100
)

print("Final Chronological Model Comparison")
print("=====================================")

display(final_model_comparison_df)

print("\nMean Performance")
print("================")

print(
    f"Ridge Mean RMSE: "
    f"{final_model_comparison_df['Ridge_RMSE'].mean():.6f} V"
)
print(
    f"XGBoost Mean RMSE: "
    f"{final_model_comparison_df['XGB_RMSE'].mean():.6f} V"
)

print(
    f"\nRidge Mean MAE: "
    f"{final_model_comparison_df['Ridge_MAE'].mean():.6f} V"
)
print(
    f"XGBoost Mean MAE: "
    f"{final_model_comparison_df['XGB_MAE'].mean():.6f} V"
)

print(
    f"\nRidge Mean R²: "
    f"{final_model_comparison_df['Ridge_R2'].mean():.6f}"
)
print(
    f"XGBoost Mean R²: "
    f"{final_model_comparison_df['XGB_R2'].mean():.6f}"
)

overall_rmse_reduction = (
    (
        final_model_comparison_df["Ridge_RMSE"].mean()
        - final_model_comparison_df["XGB_RMSE"].mean()
    )
    / final_model_comparison_df["Ridge_RMSE"].mean()
    * 100
)

print(
    f"\nMean RMSE reduction of XGBoost over Ridge: "
    f"{overall_rmse_reduction:.2f}%"
)

Final Chronological Model Comparison


,Fold,Ridge_RMSE,Ridge_MAE,Ridge_R2,XGB_RMSE,XGB_MAE,XGB_R2,RMSE_Reduction_%
0,Fold_1,0.018224,0.014882,0.965270,0.007211,0.005092,0.994562,60.431299
1,Fold_2,0.016332,0.012935,0.973288,0.009716,0.006591,0.990547,40.509429
2,Fold_3,0.019800,0.016798,0.962488,0.011420,0.009267,0.987520,42.323232
3,Fold_4,0.014051,0.010049,0.979747,0.010205,0.007668,0.989318,27.371717



Mean Performance
Ridge Mean RMSE: 0.017102 V
XGBoost Mean RMSE: 0.009638 V

Ridge Mean MAE: 0.013666 V
XGBoost Mean MAE: 0.007154 V

Ridge Mean R²: 0.970198
XGBoost Mean R²: 0.990487

Mean RMSE reduction of XGBoost over Ridge: 43.64%


In [ ]:
### F1.15.1 — Development-Stage Model Comparison and Selection

The final Ridge and XGBoost configurations were compared across the same four
expanding chronological validation folds.

Ridge Regression achieved mean RMSE = 0.01710 V, MAE = 0.01367 V and
R² = 0.97020.

XGBoost achieved mean RMSE = 0.00964 V, MAE = 0.00715 V and R² = 0.99049.

XGBoost produced lower RMSE and MAE and higher R² in every chronological
validation fold. Relative to Ridge, its mean RMSE was reduced by approximately
43.64%, with fold-wise RMSE reductions ranging from approximately 27% to 60%.

These results indicate that the relationship between the retained PEMFC
operational predictors and instantaneous stack voltage contains nonlinear
structure and/or interactions that are represented more effectively by
XGBoost than by the regularized linear Ridge model.

XGBoost is therefore selected as the primary predictive model, while Ridge is
retained as the regularized linear benchmark.

This decision is based exclusively on development-stage chronological
validation. The final 900–1000 h holdout has remained untouched and will now
be used once for final out-of-sample evaluation.

In [ ]:
## F1.16 — Final Model Training on the Complete Development Dataset

Following chronological model comparison, the final Ridge and XGBoost
configurations are now trained using the complete development period
covering durability stages from 50 h to 850 h.

Ridge is retained as the regularized linear benchmark, while XGBoost is the
primary predictive model based on its superior chronological validation
performance.

No observations from the final 900–1000 h holdout are used during this
training stage.

The resulting fitted models will subsequently be evaluated once on the
untouched final holdout to assess generalisation to later durability stages.

In [62]:
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# ---------------------------------------------------------
# 1. Prepare complete development data: 50–850 h
# ---------------------------------------------------------

X_final_development = development_df[boruta_candidate_predictors]
y_final_development = development_df["voltage"]

print("Final Development Dataset")
print("=========================")
print(f"Observations: {len(X_final_development):,}")
print(f"Predictors:   {X_final_development.shape[1]}")
print(
    f"Durability stages: "
    f"{development_df['operating_hour'].min()}–"
    f"{development_df['operating_hour'].max()} h"
)


# ---------------------------------------------------------
# 2. Final Ridge model
# ---------------------------------------------------------

final_ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(
        alpha=1.0,
        fit_intercept=True,
        solver="auto"
    ))
])

ridge_start = time.time()

final_ridge_model.fit(
    X_final_development,
    y_final_development
)

ridge_runtime = time.time() - ridge_start


# ---------------------------------------------------------
# 3. Final XGBoost model
# ---------------------------------------------------------

final_xgb_model = XGBRegressor(
    **xgb_baseline_config
)

xgb_start = time.time()

final_xgb_model.fit(
    X_final_development,
    y_final_development
)

xgb_runtime = time.time() - xgb_start


# ---------------------------------------------------------
# 4. Training summary
# ---------------------------------------------------------

print("\nFinal Model Training Completed")
print("==============================")

print(
    f"Ridge training runtime: "
    f"{ridge_runtime:.2f} seconds"
)

print(
    f"XGBoost training runtime: "
    f"{xgb_runtime:.2f} seconds"
)

print("\nStatus")
print("------")
print("Ridge:   fitted on complete 50–850 h development data")
print("XGBoost: fitted on complete 50–850 h development data")
print("900–1000 h holdout: still untouched")

Final Development Dataset
Observations: 3,049,120
Predictors:   20
Durability stages: 50–850 h

Final Model Training Completed
Ridge training runtime: 3.43 seconds
XGBoost training runtime: 77.48 seconds

Status
------
Ridge:   fitted on complete 50–850 h development data
XGBoost: fitted on complete 50–850 h development data
900–1000 h holdout: still untouched


In [ ]:
## F1.17 — Final Holdout Evaluation on 900–1000 h

The final Ridge and XGBoost models have been trained using the complete
50–850 h development dataset.

They are now evaluated once on the previously untouched 900–1000 h holdout.

This provides the final out-of-sample assessment of whether the learned
relationship between PEMFC operating variables and instantaneous stack
voltage generalises to later durability stages.

No further hyperparameter tuning or model selection will be performed using
the holdout results.

In [63]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import time

# ---------------------------------------------------------
# 1. Prepare untouched final holdout
# ---------------------------------------------------------

X_final_holdout = holdout_df[boruta_candidate_predictors]
y_final_holdout = holdout_df["voltage"]

print("Final Holdout Dataset")
print("=====================")
print(f"Observations: {len(X_final_holdout):,}")
print(f"Predictors:   {X_final_holdout.shape[1]}")
print(
    f"Durability stages: "
    f"{holdout_df['operating_hour'].min()}–"
    f"{holdout_df['operating_hour'].max()} h"
)


# ---------------------------------------------------------
# 2. Ridge prediction
# ---------------------------------------------------------

ridge_start = time.time()

ridge_holdout_pred = final_ridge_model.predict(
    X_final_holdout
)

ridge_prediction_runtime = time.time() - ridge_start

ridge_holdout_rmse = np.sqrt(
    mean_squared_error(
        y_final_holdout,
        ridge_holdout_pred
    )
)

ridge_holdout_mae = mean_absolute_error(
    y_final_holdout,
    ridge_holdout_pred
)

ridge_holdout_r2 = r2_score(
    y_final_holdout,
    ridge_holdout_pred
)


# ---------------------------------------------------------
# 3. XGBoost prediction
# ---------------------------------------------------------

xgb_start = time.time()

xgb_holdout_pred = final_xgb_model.predict(
    X_final_holdout
)

xgb_prediction_runtime = time.time() - xgb_start

xgb_holdout_rmse = np.sqrt(
    mean_squared_error(
        y_final_holdout,
        xgb_holdout_pred
    )
)

xgb_holdout_mae = mean_absolute_error(
    y_final_holdout,
    xgb_holdout_pred
)

xgb_holdout_r2 = r2_score(
    y_final_holdout,
    xgb_holdout_pred
)


# ---------------------------------------------------------
# 4. Final comparison table
# ---------------------------------------------------------

final_holdout_results_df = pd.DataFrame({
    "Model": [
        "Ridge Regression",
        "XGBoost"
    ],
    "RMSE": [
        ridge_holdout_rmse,
        xgb_holdout_rmse
    ],
    "MAE": [
        ridge_holdout_mae,
        xgb_holdout_mae
    ],
    "R2": [
        ridge_holdout_r2,
        xgb_holdout_r2
    ],
    "Prediction_Runtime_s": [
        ridge_prediction_runtime,
        xgb_prediction_runtime
    ]
})

print("\nFinal Holdout Performance")
print("=========================")

display(final_holdout_results_df)


# ---------------------------------------------------------
# 5. XGBoost improvement over Ridge
# ---------------------------------------------------------

holdout_rmse_reduction = (
    (
        ridge_holdout_rmse
        - xgb_holdout_rmse
    )
    / ridge_holdout_rmse
    * 100
)

holdout_mae_reduction = (
    (
        ridge_holdout_mae
        - xgb_holdout_mae
    )
    / ridge_holdout_mae
    * 100
)

print("\nFinal Holdout Comparison")
print("========================")

print(
    f"XGBoost RMSE reduction over Ridge: "
    f"{holdout_rmse_reduction:.2f}%"
)

print(
    f"XGBoost MAE reduction over Ridge: "
    f"{holdout_mae_reduction:.2f}%"
)

print("\nImportant:")
print(
    "The 900–1000 h holdout has now been evaluated. "
    "It must not be used for further tuning or model selection."
)

Final Holdout Dataset
Observations: 580,560
Predictors:   20
Durability stages: 900–1000 h

Final Holdout Performance


,Model,RMSE,MAE,R2,Prediction_Runtime_s
0,Ridge Regression,0.010694,0.007834,0.987909,0.105272
1,XGBoost,0.008788,0.006357,0.991834,1.619445



Final Holdout Comparison
XGBoost RMSE reduction over Ridge: 17.82%
XGBoost MAE reduction over Ridge: 18.86%

Important:
The 900–1000 h holdout has now been evaluated. It must not be used for further tuning or model selection.


In [ ]:
### F1.17.1 — Final Holdout Performance Interpretation

Both final models demonstrated strong predictive performance on the previously
untouched 900–1000 h durability holdout.

Ridge Regression achieved RMSE = 0.01069 V, MAE = 0.00783 V and
R² = 0.98791.

XGBoost achieved RMSE = 0.00879 V, MAE = 0.00636 V and
R² = 0.99183.

XGBoost therefore reduced holdout RMSE by approximately 17.82% and MAE by
approximately 18.86% relative to Ridge.

The final holdout results support the model-selection decision made using the
development-stage chronological validation, with XGBoost retaining superior
predictive performance on unseen later durability stages.

However, the performance difference between XGBoost and Ridge was smaller on
the final holdout than across the development-stage validation folds. This
indicates that both models generalised strongly to the final durability
period, while XGBoost maintained a measurable nonlinear predictive advantage.

The holdout results are treated strictly as final out-of-sample evaluation
evidence and will not be used for further hyperparameter tuning or model
selection.

In [ ]:
## F1.18 — Stage-Wise Performance Across the Final Holdout

The combined 900–1000 h holdout confirmed strong out-of-sample performance for
both final models. To examine whether this performance remains consistent
across the later durability period, the final holdout is now evaluated
separately at 900 h, 950 h and 1000 h.

RMSE, MAE and R² are calculated independently for each durability stage using
the predictions already generated by the frozen final models.

This analysis is diagnostic only. It does not involve model retraining,
hyperparameter tuning or further model selection.

In [64]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Create holdout prediction dataframe
# ---------------------------------------------------------

holdout_predictions_df = pd.DataFrame({
    "operating_hour": holdout_df["operating_hour"].to_numpy(),
    "actual_voltage": y_final_holdout.to_numpy(),
    "ridge_predicted_voltage": ridge_holdout_pred,
    "xgb_predicted_voltage": xgb_holdout_pred
})

# Residual = Actual - Predicted
holdout_predictions_df["ridge_residual"] = (
    holdout_predictions_df["actual_voltage"]
    - holdout_predictions_df["ridge_predicted_voltage"]
)

holdout_predictions_df["xgb_residual"] = (
    holdout_predictions_df["actual_voltage"]
    - holdout_predictions_df["xgb_predicted_voltage"]
)


# ---------------------------------------------------------
# Calculate performance separately for each holdout stage
# ---------------------------------------------------------

stage_results = []

for stage in sorted(holdout_predictions_df["operating_hour"].unique()):

    stage_df = holdout_predictions_df[
        holdout_predictions_df["operating_hour"] == stage
    ]

    # Ridge
    ridge_rmse = np.sqrt(
        mean_squared_error(
            stage_df["actual_voltage"],
            stage_df["ridge_predicted_voltage"]
        )
    )

    ridge_mae = mean_absolute_error(
        stage_df["actual_voltage"],
        stage_df["ridge_predicted_voltage"]
    )

    ridge_r2 = r2_score(
        stage_df["actual_voltage"],
        stage_df["ridge_predicted_voltage"]
    )

    # XGBoost
    xgb_rmse = np.sqrt(
        mean_squared_error(
            stage_df["actual_voltage"],
            stage_df["xgb_predicted_voltage"]
        )
    )

    xgb_mae = mean_absolute_error(
        stage_df["actual_voltage"],
        stage_df["xgb_predicted_voltage"]
    )

    xgb_r2 = r2_score(
        stage_df["actual_voltage"],
        stage_df["xgb_predicted_voltage"]
    )

    stage_results.append({
        "Operating_Hour": stage,
        "Observations": len(stage_df),

        "Ridge_RMSE": ridge_rmse,
        "Ridge_MAE": ridge_mae,
        "Ridge_R2": ridge_r2,

        "XGB_RMSE": xgb_rmse,
        "XGB_MAE": xgb_mae,
        "XGB_R2": xgb_r2
    })


stage_holdout_results_df = pd.DataFrame(stage_results)

# ---------------------------------------------------------
# XGBoost RMSE improvement over Ridge at each stage
# ---------------------------------------------------------

stage_holdout_results_df["XGB_RMSE_Reduction_%"] = (
    (
        stage_holdout_results_df["Ridge_RMSE"]
        - stage_holdout_results_df["XGB_RMSE"]
    )
    / stage_holdout_results_df["Ridge_RMSE"]
    * 100
)


print("Stage-Wise Final Holdout Performance")
print("====================================")

display(
    stage_holdout_results_df.round(6)
)

Stage-Wise Final Holdout Performance


,Operating_Hour,Observations,Ridge_RMSE,Ridge_MAE,Ridge_R2,XGB_RMSE,XGB_MAE,XGB_R2,XGB_RMSE_Reduction_%
0,900,179360,0.009684,0.007224,0.990241,0.006034,0.004596,0.996210,37.685572
1,950,179360,0.011417,0.007974,0.985976,0.008012,0.005976,0.993094,29.824354
2,1000,221840,0.010866,0.008214,0.987525,0.010990,0.008089,0.987238,-1.146004


In [ ]:
### F1.18.1 — Interpretation of Stage-Wise Holdout Performance

Stage-wise evaluation showed that predictive performance varied across the
three unseen durability stages.

At 900 h, XGBoost achieved RMSE = 0.00603 V and R² = 0.99621, reducing RMSE
by approximately 37.69% relative to Ridge. At 950 h, XGBoost remained
superior, achieving RMSE = 0.00801 V and R² = 0.99309, corresponding to an
RMSE reduction of approximately 29.82%.

At 1000 h, however, the difference between the models became very small.
Ridge achieved RMSE = 0.01087 V compared with 0.01099 V for XGBoost, while
XGBoost retained a slightly lower MAE (0.00809 V compared with 0.00821 V).

Therefore, XGBoost demonstrated a clear predictive advantage at 900 h and
950 h, while the two models showed broadly comparable performance at 1000 h.
The stage-wise results also show that the magnitude of XGBoost's advantage
was not constant across the final holdout period.

The 1000 h result motivates further examination of prediction residuals and
error distributions. No additional model tuning or model selection is
performed from these diagnostic results.

In [ ]:
## F1.19 — Final Holdout Residual Diagnostics

Residual diagnostics are performed on the frozen final-model predictions from
the 900–1000 h holdout.

Residuals are defined as:

Residual = Actual Voltage − Predicted Voltage

A residual close to zero indicates accurate prediction. A positive mean
residual indicates systematic underprediction of voltage, while a negative
mean residual indicates systematic overprediction.

Residual distributions are examined separately at 900 h, 950 h and 1000 h to
assess whether prediction errors change across the unseen later durability
stages.

This analysis is diagnostic only and does not influence model training,
hyperparameter tuning or model selection.

In [65]:
# ---------------------------------------------------------
# F1.19.1 — Stage-wise residual summary
# ---------------------------------------------------------

residual_summary = []

for stage in sorted(holdout_predictions_df["operating_hour"].unique()):

    stage_df = holdout_predictions_df[
        holdout_predictions_df["operating_hour"] == stage
    ]

    for model, residual_column in [
        ("Ridge", "ridge_residual"),
        ("XGBoost", "xgb_residual")
    ]:

        residuals = stage_df[residual_column]

        residual_summary.append({
            "Operating_Hour": stage,
            "Model": model,
            "Mean_Residual": residuals.mean(),
            "Median_Residual": residuals.median(),
            "Residual_SD": residuals.std(),
            "Min_Residual": residuals.min(),
            "Max_Residual": residuals.max(),
            "P05": residuals.quantile(0.05),
            "P95": residuals.quantile(0.95)
        })


residual_summary_df = pd.DataFrame(residual_summary)

print("Stage-Wise Residual Summary")
print("===========================")

display(
    residual_summary_df.round(6)
)

Stage-Wise Residual Summary


,Operating_Hour,Model,Mean_Residual,Median_Residual,Residual_SD,Min_Residual,Max_Residual,P05,P95
0,900,Ridge,-0.001476,-0.000518,0.009570,-0.115720,0.048336,-0.016949,0.012263
1,900,XGBoost,0.003172,0.003226,0.005133,-0.054710,0.062338,-0.003883,0.010381
2,950,Ridge,-0.000920,-0.000042,0.011380,-0.360713,0.067465,-0.020047,0.015384
3,950,XGBoost,0.004084,0.003488,0.006893,-0.103002,0.105176,-0.005878,0.015244
4,1000,Ridge,0.001234,0.002514,0.010796,-0.568249,0.075330,-0.017337,0.016625
5,1000,XGBoost,0.003295,0.004155,0.010485,-0.360099,0.186361,-0.014212,0.017786


In [ ]:
### F1.19.1 — Interpretation of Residual Behaviour

Residual analysis showed that XGBoost exhibited a small positive mean residual
across all three final holdout stages, corresponding to a slight tendency to
underpredict observed stack voltage. Mean residuals were approximately
3.17 mV at 900 h, 4.08 mV at 950 h and 3.30 mV at 1000 h.

The more notable change occurred in residual dispersion. XGBoost residual
standard deviation increased from 0.00513 V at 900 h to 0.00689 V at 950 h
and 0.01049 V at 1000 h. This indicates that the reduction in predictive
performance at later holdout stages was associated primarily with increased
error variability rather than a progressively increasing systematic bias.

At 1000 h, the central residual distribution remained relatively narrow, with
the 5th and 95th percentiles at approximately -0.0142 V and +0.0178 V,
respectively. However, substantially larger individual residuals were also
observed. These larger errors help explain why XGBoost produced a slightly
higher RMSE than Ridge at 1000 h despite retaining a slightly lower MAE.

Extreme residuals were present for both models and should not be interpreted
as degradation effects without examining their relationship with operating
conditions and the experimental sequence.

Overall, the residual analysis confirms strong general predictive performance
while showing that prediction-error variability increases in the later
durability stages.

In [ ]:
### F1.19.2 — Actual vs Predicted Voltage

An actual-versus-predicted plot is used to visually assess the agreement
between measured stack voltage and XGBoost predictions across the final
900–1000 h holdout.

Because the holdout contains more than 580,000 observations, a reproducible
random sample is used for visualization to avoid excessive point overlap.
All reported performance metrics remain based on the complete holdout dataset.

Predictions close to the 1:1 reference line indicate good agreement between
measured and predicted voltage, while systematic departures from this line
would indicate prediction bias or regions of reduced model accuracy.

In [66]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Reproducible sample for visualization only
# ---------------------------------------------------------

plot_sample = holdout_predictions_df.sample(
    n=min(20000, len(holdout_predictions_df)),
    random_state=42
)

# ---------------------------------------------------------
# Actual vs predicted XGBoost voltage
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    plot_sample["actual_voltage"],
    plot_sample["xgb_predicted_voltage"],
    alpha=0.25,
    s=10
)

# 1:1 reference limits
voltage_min = min(
    plot_sample["actual_voltage"].min(),
    plot_sample["xgb_predicted_voltage"].min()
)

voltage_max = max(
    plot_sample["actual_voltage"].max(),
    plot_sample["xgb_predicted_voltage"].max()
)

ax.plot(
    [voltage_min, voltage_max],
    [voltage_min, voltage_max],
    linestyle="--",
    linewidth=1.5
)

ax.set_xlim(voltage_min, voltage_max)
ax.set_ylim(voltage_min, voltage_max)

ax.set_xlabel("Actual Voltage (V)")
ax.set_ylabel("Predicted Voltage (V)")
ax.set_title(
    "XGBoost: Actual vs Predicted Voltage\n"
    "Final Holdout (900–1000 h)"
)

ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

<Figure size 700x600 with 1 Axes>

In [ ]:
### F1.19.2.1 — Interpretation of Actual vs Predicted Voltage

The actual-versus-predicted plot demonstrates strong overall agreement between
measured stack voltage and XGBoost predictions across the final 900–1000 h
holdout. Most observations are concentrated close to the 1:1 reference line,
consistent with the high holdout R² of 0.99183 and low RMSE of 0.00879 V.

Several dense voltage regions are visible, reflecting the different operating
regimes represented within the dynamic durability data. Prediction dispersion
is somewhat greater within parts of the intermediate voltage range than in
some of the more tightly clustered voltage regions.

A small number of observations also show substantially larger deviations from
the 1:1 line. This agrees with the residual statistics, which showed that the
central residual distribution is relatively narrow while occasional larger
prediction errors occur.

Overall, the plot supports strong generalisation of the final XGBoost model
to the unseen later durability stages while also indicating that prediction
accuracy varies across operating regions.

In [ ]:
### F1.19.3 — Stage-Wise XGBoost Residual Distribution

The distribution of XGBoost residuals is examined separately at 900 h,
950 h and 1000 h to assess how prediction-error behaviour changes across
the unseen durability stages.

Residual = Actual Voltage − Predicted Voltage

A residual of zero represents an exact prediction. Positive residuals indicate
voltage underprediction, while negative residuals indicate overprediction.

The visualization focuses on the central residual distribution so that the
typical error behaviour can be compared without the plot being dominated by
the relatively small number of extreme residuals.

In [67]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# F1.19.3 — Stage-wise XGBoost residual distributions
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

stages = sorted(
    holdout_predictions_df["operating_hour"].unique()
)

residual_data = [
    holdout_predictions_df.loc[
        holdout_predictions_df["operating_hour"] == stage,
        "xgb_residual"
    ]
    for stage in stages
]

# Boxplot
ax.boxplot(
    residual_data,
    tick_labels=[f"{int(stage)} h" for stage in stages],
    showfliers=False
)

# Zero-residual reference line
ax.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Durability Stage")
ax.set_ylabel("Residual (V)")
ax.set_title(
    "XGBoost Residual Distribution by Durability Stage\n"
    "Final Holdout (900–1000 h)"
)

ax.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

In [ ]:
### F1.19.3.1 — Interpretation of Stage-Wise Residual Distributions

The stage-wise boxplots show a progressive increase in the spread of XGBoost
residuals from 900 h to 1000 h. The residual distribution is relatively narrow
at 900 h, becomes broader at 950 h, and is widest at 1000 h.

This pattern agrees with the increase in residual standard deviation from
0.00513 V at 900 h to 0.00689 V at 950 h and 0.01049 V at 1000 h.
Therefore, the reduction in predictive accuracy across the later holdout
stages is associated primarily with increasing error variability.

The residual medians remain slightly above zero at all three stages,
indicating a small tendency for XGBoost to underpredict measured stack
voltage. However, the magnitude of this bias changes relatively little
compared with the increase in residual spread.

Extreme residual observations are not displayed individually in this figure
to preserve visibility of the central distributions, but they remain included
in all numerical performance metrics and were quantified separately in the
residual summary.

Overall, the diagnostic evidence indicates that XGBoost maintains a small
underprediction bias while prediction uncertainty becomes greater across the
later unseen durability stages.

In [ ]:
### F1.19.4 — XGBoost Residuals Across the Holdout Sequence

XGBoost residuals are examined across the sequential observations within each
final holdout stage to determine whether prediction errors are distributed
throughout the dynamic operating sequence or concentrated within particular
regions.

Because each stage contains a very large number of observations, the residual
series is downsampled at regular intervals for visualization only. All
performance metrics and residual statistics remain based on the complete
holdout dataset.

This diagnostic can reveal temporal or operating-sequence structure in model
errors but does not, by itself, establish the physical cause of those errors.

In [68]:
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# F1.19.4 — Residuals across holdout sequence
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))

stages = sorted(
    holdout_predictions_df["operating_hour"].unique()
)

sequence_offset = 0
stage_boundaries = []
stage_centres = []

for stage in stages:

    stage_df = holdout_predictions_df[
        holdout_predictions_df["operating_hour"] == stage
    ].reset_index(drop=True)

    # Sequential position within the stage
    stage_sequence = np.arange(len(stage_df))

    # Regular downsampling for visualization only
    step = max(1, len(stage_df) // 5000)

    sampled_sequence = stage_sequence[::step]
    sampled_residuals = stage_df["xgb_residual"].to_numpy()[::step]

    global_sequence = sampled_sequence + sequence_offset

    ax.scatter(
        global_sequence,
        sampled_residuals,
        s=5,
        alpha=0.35,
        label=f"{int(stage)} h"
    )

    # Store stage centre for x-axis labels
    stage_centres.append(
        sequence_offset + len(stage_df) / 2
    )

    sequence_offset += len(stage_df)

    # Store boundary between stages
    stage_boundaries.append(sequence_offset)


# ---------------------------------------------------------
# Zero-residual reference
# ---------------------------------------------------------

ax.axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

# ---------------------------------------------------------
# Stage boundaries
# ---------------------------------------------------------

for boundary in stage_boundaries[:-1]:
    ax.axvline(
        x=boundary,
        linestyle=":",
        linewidth=1
    )


# ---------------------------------------------------------
# Formatting
# ---------------------------------------------------------

ax.set_xticks(stage_centres)

ax.set_xticklabels(
    [f"{int(stage)} h" for stage in stages]
)

ax.set_xlabel("Sequential Position Within Final Holdout")
ax.set_ylabel("Residual (V)")

ax.set_title(
    "XGBoost Residuals Across the Final Holdout Sequence\n"
    "(900–1000 h)"
)

ax.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()
plt.show()

<Figure size 1200x500 with 1 Axes>

In [ ]:
### F1.19.4.1 — Interpretation of Sequential Residual Behaviour

Residuals across the final holdout show clear sequential structure rather than
uniform random dispersion around zero.

At 900 h, XGBoost residuals remain comparatively tightly concentrated around
zero across most of the dynamic sequence, consistent with the strong
stage-specific predictive performance.

At 950 h, distinct regions of positive and negative residual behaviour become
more apparent, indicating that prediction error varies across different
portions of the operating sequence.

The 1000 h stage exhibits the greatest residual variability. Larger positive
and negative excursions occur within particular regions of the sequence rather
than uniformly across the entire stage. This indicates that the reduction in
predictive accuracy at 1000 h is associated with specific portions of the
dynamic operating sequence.

The sequential structure suggests that model errors may be related to changes
in operating regime, transient behaviour or later-stage system behaviour.
However, residual structure alone cannot identify the underlying physical
cause and is therefore not interpreted directly as evidence of PEMFC
degradation.

These results will later be considered alongside operating-condition behaviour
and polarization-curve evidence when interpreting degradation.

In [ ]:
## F1.20 — Save Final Modelling Artefacts

The final modelling outputs are saved to provide a reproducible interface
between model development and subsequent degradation analysis.

Saved artefacts include the final holdout predictions and residuals,
overall holdout performance, stage-wise holdout performance, chronological
model-comparison results and the retained model configurations.

The original processed dataset is not duplicated because it can be loaded
directly in subsequent notebooks when operational variables are required.

In [69]:
from pathlib import Path
import json
import joblib

# ---------------------------------------------------------
# 1. Output directories
# ---------------------------------------------------------

results_dir = Path("../results/notebook_12")
models_dir = Path("../models")

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 2. Save final holdout predictions and residuals
# ---------------------------------------------------------

holdout_predictions_df.to_csv(
    results_dir / "final_holdout_predictions.csv",
    index=False
)


# ---------------------------------------------------------
# 3. Save overall final holdout metrics
# ---------------------------------------------------------

final_holdout_results_df.to_csv(
    results_dir / "final_holdout_metrics.csv",
    index=False
)


# ---------------------------------------------------------
# 4. Save stage-wise holdout performance
# ---------------------------------------------------------

stage_holdout_results_df.to_csv(
    results_dir / "stage_wise_holdout_metrics.csv",
    index=False
)


# ---------------------------------------------------------
# 5. Save chronological Ridge vs XGBoost comparison
# ---------------------------------------------------------

final_model_comparison_df.to_csv(
    results_dir / "chronological_model_comparison.csv",
    index=False
)


# ---------------------------------------------------------
# 6. Save residual summary
# ---------------------------------------------------------

residual_summary_df.to_csv(
    results_dir / "stage_wise_residual_summary.csv",
    index=False
)


# ---------------------------------------------------------
# 7. Save final model configurations
# ---------------------------------------------------------

final_model_configurations = {
    "Ridge": {
        "alpha": 1.0,
        "fit_intercept": True,
        "solver": "auto",
        "scaling": "StandardScaler"
    },

    "XGBoost": xgb_baseline_config,

    "predictors": boruta_candidate_predictors,

    "development_stages": [
        int(x)
        for x in sorted(
            development_df["operating_hour"].unique()
        )
    ],

    "final_holdout_stages": [
        int(x)
        for x in sorted(
            holdout_df["operating_hour"].unique()
        )
    ]
}

with open(
    results_dir / "final_model_configurations.json",
    "w"
) as f:
    json.dump(
        final_model_configurations,
        f,
        indent=4
    )


# ---------------------------------------------------------
# 8. Save fitted final models
# ---------------------------------------------------------

joblib.dump(
    final_ridge_model,
    models_dir / "final_ridge_model.joblib"
)

joblib.dump(
    final_xgb_model,
    models_dir / "final_xgboost_model.joblib"
)


# ---------------------------------------------------------
# 9. Confirm saved artefacts
# ---------------------------------------------------------

print("Final Modelling Artefacts Saved")
print("================================")

print("\nResults:")
for file in sorted(results_dir.iterdir()):
    print(f"  {file.name}")

print("\nModels:")
print("  final_ridge_model.joblib")
print("  final_xgboost_model.joblib")

Final Modelling Artefacts Saved

Results:
  chronological_model_comparison.csv
  final_holdout_metrics.csv
  final_holdout_predictions.csv
  final_model_configurations.json
  stage_wise_holdout_metrics.csv
  stage_wise_residual_summary.csv

Models:
  final_ridge_model.joblib
  final_xgboost_model.joblib


In [ ]:
## F1.21 — Notebook 12 Summary

This notebook developed and evaluated Ridge Regression and XGBoost models for
predicting instantaneous PEMFC stack voltage from 20 operational and
engineered predictors.

Model development was restricted to durability stages from 50 h to 850 h.
Four expanding chronological validation folds were used to evaluate
generalisation to progressively later durability stages while preserving the
temporal structure of the experiment.

Ridge Regression was used as the regularized linear benchmark. Nested
chronological tuning showed that changing the Ridge regularization strength
produced negligible improvement, and α = 1.0 was therefore retained.

XGBoost was evaluated as the nonlinear predictive model. Nested chronological
hyperparameter tuning did not consistently improve outer-fold performance over
the baseline configuration. The baseline XGBoost configuration was therefore
retained.

Across the four chronological development folds, Ridge achieved mean
RMSE = 0.01710 V, MAE = 0.01367 V and R² = 0.97020. XGBoost achieved mean
RMSE = 0.00964 V, MAE = 0.00715 V and R² = 0.99049. XGBoost reduced mean
RMSE by approximately 43.64% relative to Ridge and was selected as the primary
predictive model.

Following model selection, both models were retrained using the complete
50–850 h development dataset and evaluated once on the previously untouched
900–1000 h final holdout.

On the final holdout, Ridge achieved RMSE = 0.01069 V, MAE = 0.00783 V and
R² = 0.98791. XGBoost achieved RMSE = 0.00879 V, MAE = 0.00636 V and
R² = 0.99183, corresponding to reductions of approximately 17.82% in RMSE
and 18.86% in MAE relative to Ridge.

Stage-wise evaluation showed that XGBoost clearly outperformed Ridge at
900 h and 950 h, while the two models produced broadly comparable performance
at 1000 h. Residual diagnostics showed a small XGBoost underprediction bias
and increasing residual variability across the later holdout stages.
Sequential residual analysis further showed that larger errors were
concentrated within particular portions of the dynamic operating sequence
rather than being uniformly distributed.

Overall, the results demonstrate that both models generalise strongly to
unseen later durability stages, while XGBoost provides the strongest overall
predictive performance. The improvement over the linear Ridge benchmark
indicates that nonlinear relationships and/or interactions contribute
meaningfully to the mapping between the retained operational predictors and
instantaneous stack voltage.

The modelling results describe predictive behaviour and are not interpreted
alone as direct evidence of PEMFC degradation mechanisms. Subsequent analysis
will combine these modelling outputs with durability-stage behaviour and
polarization-curve evidence to investigate PEMFC performance degradation and
recovery.

Final predictions, residuals, performance metrics, model configurations and
fitted models have been saved for subsequent analysis.